# 실제 Photoneo 촬영 데이터 검증: 시뮬레이션으로 학습한 side plate 모델 (70장)

시뮬레이션(Isaac Sim)으로만 학습한 파이프라인을 실제 Photoneo 촬영 **70장**(`data/pseudo_gt_70_bop`, 부품 390개)에 적용하고, 의사 정답(pseudo-GT)과 비교합니다.
파이프라인은 다음 순서로 진행됩니다.

**Mask R-CNN detector → GDRN 방식 RGB-D pose 네트워크 → 깊이 맞춤 → ICP → 중복 제거 → 깊이 검증(오검출 제거) → GT 없는 confidence 판정**

**사용한 모델:** `runs/detector/best.pt`, `runs/pose/best.pt`. 둘 다 side plate 시뮬레이션 데이터로만 학습했습니다. 이 평가를 위해 재학습한 것은 없습니다.

**정답에 대한 주의:** 정답은 FoundationPose가 추정한 **의사 정답**입니다. 아래 오차는 진짜 오차가 아니라 **FoundationPose 결과와의 차이**입니다.
의사 정답 자체에도 카메라 방향 위치 약 0.5mm, 기울기 1~2° 수준의 불확실성이 있습니다(`data/pseudo_gt_70_bop/README.txt`). 그래서 0.5mm 아래의 차이는 구분할 수 없습니다.

**두 묶음으로 나눠서 봅니다.** 70장 중 0~19번은 이전 분석에 쓴 20장과 같은 촬영입니다. 깊이 검증 기준은 이 20장(과 시뮬레이션)을 보고 정했습니다.
**20~69번 50장은 기준을 정할 때 보지 않은 데이터**라서, 기준이 우연히 맞춰진 것이 아닌지 확인하는 데 씁니다.

§6에서는 detector를 실제 데이터로 fine-tuning한 결과를 비교합니다. §7에서는 정답이 없는 가득 찬 실제 bin(`data/real_capture_bop`)에 적용하고, §8에서는 pose 네트워크를 fine-tuning한 결과를 비교합니다. 결론은 §9에 정리했습니다. §10에는 부품별 6D pose 표(위치, 회전, 허브 축, depth 확인, 덮이지 않은 부품만 고른 pick 순서), 맨 위이면서 confidence가 높고 가운데 구멍 주변이 비어 있는(내경 그리퍼로 집을 수 있는) 10개의 6D pose와 집는 점 출력, 단계별 정확도 표, CSV 저장이 있습니다. 모든 숫자는 `runs/`의 추론 결과에서 매번 다시 계산합니다.

In [ ]:
import os, subprocess, sys
from pathlib import Path
if Path.cwd().name == "notebooks":                      # run from the project root
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import json
import numpy as np, pandas as pd, plotly.graph_objects as go
from IPython.display import display
from PIL import Image

from data_conversion.bin_dataset import BinPoseDataset
from bin6d.config import Cfg, load_config
from bin6d.pose_data import ObjectModel
from bin6d.report import (accuracy_table, detections_table, frame_index, load_frame, load_run, per_frame_table)
from bin6d.viz import gt_overlay_sets, plotly_scene, render_2d, select

cfg = load_config("configs/default.yaml")
REAL_ROOT = "data/pseudo_gt_70_bop"
DET, GTM = "detector masks (real pipeline)", "ground-truth masks"
RUNS = {DET: "runs/photoneo70_full", GTM: "runs/photoneo70_gtmask"}                       # current pipeline
RUNS_NOREJECT = {DET: "runs/photoneo70_full_noreject"}                                    # same, without verification
RUNS_BEFORE = {DET: "runs/photoneo70_full_noanchor", GTM: "runs/photoneo70_gtmask_noanchor"}  # no anchor/dedup/verification
SIM_RUNS = {DET: "runs/eval_full_verified", GTM: "runs/eval_gtmask_verified"}            # sim, current pipeline
SIM_ANCHOR = {DET: "runs/eval_full_anchor", GTM: "runs/eval_gtmask_anchor"}              # sim, anchor + dedup only
SIM_BEFORE = {DET: "runs/eval_full", GTM: "runs/eval_gtmask"}
DET_SIM, DET_FT = "runs/detector/best.pt", "runs/detector_ft/best.pt"                   # sim only / sim + real (§6)
RUNS_FT = {DET: "runs/photoneo70_ft"}                                                       # fine-tuned detector, current pipeline
RUNS_FT_NOREJECT = {DET: "runs/photoneo70_ft_noreject"}
SIM_RUNS_FT = {DET: "runs/eval_full_ft"}
TUNED, NEW = "first 20 (thresholds tuned here)", "new 50 (unseen)"                         # views 0-19 = the 20-frame set
real = BinPoseDataset(REAL_ROOT)
SCENE = real.frames[0].scene
ridx = frame_index(real)
model = ObjectModel(real, cfg.data.obj_id, 0, 3000)
manifest = json.load(open(Path(REAL_ROOT) / "manifest.json"))
N_PARTS = sum(len(real.frame_meta(i)["objects_all"]["t"]) for i in range(len(real)))
pd.set_option("display.width", 200)
OVERLAP = set(manifest["overlap_views"])
PARTS = {v: len(real.frame_meta(ridx[(SCENE, v)])["objects_all"]["t"]) for v in range(len(real))}

def subset(view):
    return TUNED if view < 20 else NEW

def load_rejected(run_dir):
    """{(scene, view): [rejected detections]} written by infer.py with refine.reject_unverified."""
    out = {}
    for f in sorted((Path(run_dir) / "rejected").glob("*_*.json")):
        scene, view = f.stem.split("_")
        out[(scene, int(view))] = json.load(open(f))
    return out

# simulation held-out scene, for side-by-side numbers (skipped if the sim data is not on this machine)
try:
    VAL = cfg.data.val_scenes[0]
    sim = BinPoseDataset(cfg.data.root, scenes=[VAL])
    sidx = frame_index(sim)
except (FileNotFoundError, ValueError) as e:
    sim = None
    print(f"simulation data not available ({e}); sim comparisons are skipped")

S1, S2, S3 = "#2a78d6", "#eb6834", "#1baf7a"
def style(fig, title, height=360):
    fig.update_layout(template="plotly_white", title=title, height=height, font=dict(color="#52514e"),
                      margin=dict(l=60, r=20, t=60, b=40), legend=dict(orientation="h", y=1.12, x=0))
    fig.update_xaxes(gridcolor="#e1e0d9", linecolor="#c3c2b7")
    fig.update_yaxes(gridcolor="#e1e0d9", linecolor="#c3c2b7")
    return fig
print(f"real: {len(real)} frames ({real.split}), {N_PARTS} parts, part diameter {model.diameter:.1f} mm"
      + (f" | sim val scene {VAL}: {len(sim)} frames" if sim else ""))
print(f"refine: depth_anchor={cfg.refine.depth_anchor}, dedup_mm={cfg.refine.dedup_mm}, "
      f"reject_unverified={cfg.refine.reject_unverified} (mask cover >= {cfg.refine.verify_min_mask_cover}, "
      f"overlap < {cfg.refine.verify_max_mask_overlap})")

## 1. 데이터

| | |
|---|---|
| 형식 | BOP (`train_real/000000/rgb, depth, mask_visib_inst, scene_gt*.json`), **70장, 부품 390개** |
| 장당 부품 | 0~49번은 5개, 50~69번은 2~10개 |
| 센서 | Photoneo, 1236×1032 (시뮬레이션과 해상도와 초점거리가 같음). depth는 0.1mm 단위로 저장되지만 원본이 **1mm 단위**라 값이 모두 10의 배수 |
| 정답 | FoundationPose pose (카메라 좌표계, CAD 원점 기준). CAD 원점과 크기는 시뮬레이션 모델과 같음 |
| 마스크 | SAM 마스크가 저장되지 않아 pose와 depth로 다시 만든 보이는 영역 마스크 |
| 장면 | 파란 타공 bin. **겹치거나 기대어 놓인 장면 28장**, 나머지는 떨어져 놓임. 자세는 면이 위 192개, 뒤집힘 198개 |

시뮬레이션은 같은 bin에 **부품이 가득 쌓인 장면(최대 약 300개)**을 비워 가며 찍었습니다. 실제 데이터는 훨씬 드문드문합니다.
그 대신 bin 바닥의 타공 구멍이 모두 드러나고, bin 바깥 배경(골판지, 케이블, 상자)도 보입니다.

In [ ]:
pv = pd.DataFrame(manifest["per_view"])
pv["overlapping"] = pv.view.isin(OVERLAP)
pv["min visib_fract"] = pv.visib_fract.map(min)
pv["subset"] = pv.view.map(subset)
display(pv.groupby(["subset", "overlapping"]).agg(views=("view", "size"), parts=("parts", "sum"),
                                                  parts_per_view=("parts", lambda s: f"{s.min()}-{s.max()}"),
                                                  min_visib_fract=("min visib_fract", "min")))

def strip(f, depth_range=(550, 850), h=300):
    lo, hi = depth_range
    dv = np.clip((f["depth"] - lo) / (hi - lo), 0, 1)
    dv = (np.where(f["valid"], 1 - dv, 0) * 255).astype(np.uint8)
    rng = np.random.default_rng(0); lut = rng.integers(40, 255, (f["inst"].max() + 1, 3)); lut[0] = 0
    tiles = [Image.fromarray(f["rgb"]), Image.fromarray(dv).convert("RGB"), Image.fromarray(lut[f["inst"]].astype(np.uint8))]
    w = int(h * f["rgb"].shape[1] / f["rgb"].shape[0])
    out = Image.new("RGB", (3 * w, h))
    for i, t in enumerate(tiles):
        out.paste(t.resize((w, h)), (w * i, 0))
    return out
for v in (0, 16, 59):
    print(f"view {v} ({PARTS[v]} parts): RGB | depth (black = no depth) | pseudo-GT visible masks")
    display(strip(load_frame(real, ridx[(SCENE, v)])))

### 시뮬레이션과 실제의 차이 (부품 픽셀 기준)

In [ ]:
def frame_stats(frames, i):
    f = load_frame(frames, i)
    o, part = f["objects_all"], f["inst"] > 0
    vals = np.unique(f["depth"][f["valid"]])
    return {"camera -> part distance (mm)": float(np.median(o["t"][:, 2])),
            "visible part size (px, median)": float(np.median(o["visib_px"][o["visible"]])),
            "part pixels with depth": float((part & f["valid"]).sum() / part.sum()),
            "part brightness (0-255)": float(f["rgb"][part].mean()),
            "depth step (mm)": float(np.median(np.diff(vals))),
            "parts per frame": len(o["t"])}

cols = {f"real ({len(real)} frames)": pd.DataFrame([frame_stats(real, i) for i in range(len(real))]).median()}
if sim:
    cols[f"sim (scene {VAL}, 10 frames)"] = pd.DataFrame([frame_stats(sim, sidx[(VAL, v)]) for v in range(0, 100, 10)]).median()
display(pd.DataFrame(cols).round(3))

카메라와 부품 사이 거리는 둘 다 약 700mm로 **거의 같습니다**. 보이는 부품 크기가 실제에서 더 큰 것은 가려지지 않은 부품이 많기 때문입니다.
눈에 띄는 차이는 세 가지입니다.
- **depth 단위:** 실제는 1mm, 시뮬레이션은 0.1mm입니다.
- **장면 구성:** 실제는 타공 바닥 위에 부품이 2~10개 놓여 있고, 시뮬레이션은 부품이 쌓여 있습니다.
- **밝기:** 실제 부품이 더 어둡습니다.

## 2. 추론 실행

`infer.py`를 실행해 결과(JSON)를 `runs/`에 저장합니다. §1~§5는 시뮬레이션으로만 학습한 detector(`runs/detector/best.pt`) 결과이고, §6에서 fine-tuning한 detector와 비교합니다. 결과가 이미 있으면 다시 실행하지 않습니다. 모델이나 설정을 바꿨다면 `RERUN = True`로 바꾸세요.
깊이 검증으로 걸러진 검출은 각 결과 폴더의 `rejected/`에 저장됩니다.

| 결과 폴더 | 데이터 | 마스크 | 깊이 맞춤, 중복 제거 | 깊이 검증 | 시간 (GB10) |
|---|---|---|---|---|---|
| `runs/photoneo70_full`, `runs/photoneo70_gtmask` | 실제 70장 | detector / GT | 켬 | 켬 (현재 설정) | 약 4분씩 |
| `runs/photoneo70_full_noreject` | 실제 70장 | detector | 켬 | 끔 (비교용) | 약 4분 |
| `runs/photoneo70_*_noanchor` | 실제 70장 | detector / GT | 끔 | 끔 (비교용) | 약 4분씩 |
| `runs/photoneo70_ft`, `runs/photoneo70_ft_noreject` | 실제 70장 | **fine-tuning한 detector** (§6) | 켬 | 켬 / 끔 | 약 4분씩 |
| `runs/eval_full_ft` | 시뮬레이션 검증 장면 20장 | **fine-tuning한 detector** (§6) | 켬 | 켬 | 약 10분 |
| `runs/eval_*_verified` | 시뮬레이션 검증 장면 20장 | detector / GT | 켬 | 켬 (현재 설정) | 약 10분씩 |
| `runs/eval_*_anchor` | 시뮬레이션 검증 장면 20장 | detector / GT | 켬 | 끔 | 약 10분씩 |
| `runs/eval_full`, `runs/eval_gtmask` | 시뮬레이션 검증 장면 20장 | detector / GT | 끔 | 끔 (기존 결과) | |

```bash
.venv/bin/python infer.py --set data.root=data/pseudo_gt_70_bop --scenes 000000 --view-stride 1 --no-viz --out runs/photoneo70_full
.venv/bin/python infer.py --set data.root=data/pseudo_gt_70_bop --scenes 000000 --view-stride 1 --no-viz --out runs/photoneo70_gtmask --gt-masks
```

In [ ]:
RERUN = False
REAL = ["--set", f"data.root={REAL_ROOT}", "--scenes", SCENE, "--view-stride", "1", "--detector", DET_SIM]
NOREJECT = ["--set", "refine.reject_unverified=false"]
OFF = ["--set", "refine.depth_anchor=false", "--set", "refine.dedup_mm=0"] + NOREJECT
jobs = [(RUNS[DET], REAL), (RUNS[GTM], REAL + ["--gt-masks"]), (RUNS_NOREJECT[DET], REAL + NOREJECT),
        (RUNS_BEFORE[DET], REAL + OFF), (RUNS_BEFORE[GTM], REAL + OFF + ["--gt-masks"])]
if sim:
    SIM = ["--view-stride", "5", "--detector", DET_SIM]
    jobs += [(SIM_RUNS[DET], SIM), (SIM_RUNS[GTM], SIM + ["--gt-masks"]),
             (SIM_ANCHOR[DET], SIM + NOREJECT), (SIM_ANCHOR[GTM], SIM + NOREJECT + ["--gt-masks"]),
             (SIM_BEFORE[DET], SIM + OFF), (SIM_BEFORE[GTM], SIM + OFF + ["--gt-masks"])]
FT = [a if a != DET_SIM else DET_FT for a in REAL]                  # same, with the fine-tuned detector (§6)
jobs += [(RUNS_FT[DET], FT), (RUNS_FT_NOREJECT[DET], FT + NOREJECT)]
if sim:
    jobs.append((SIM_RUNS_FT[DET], [a if a != DET_SIM else DET_FT for a in SIM]))
for out, args in jobs:
    if RERUN or not list(Path(out).glob("*_*.json")):
        cmd = [sys.executable, "infer.py", "--out", out, "--no-viz"] + args
        print(" ".join(cmd))
        subprocess.run(cmd, check=True, env={**os.environ, "HF_HUB_OFFLINE": "1"})
    print(f"{out}: {len(list(Path(out).glob('*_*.json')))} frames")

## 3. 의사 정답 대비 정확도

**ADD-S**는 예측 pose와 정답 pose에서 CAD 표면 점들 사이의 평균 거리입니다(대칭 중 최솟값). **centre err**는 위치 오차, **tilt**는 판 법선 사이의 각도입니다.
`n`은 정답과 매칭된 검출 수입니다(마스크 IoU ≥ 0.5). 5.6mm는 부품 지름의 10%로, BOP 표준 기준입니다.
**+ depth anchor** 줄은 네트워크 pose를 실측 depth에 맞춘 뒤, ICP 전의 결과입니다.

In [ ]:
tables = {name: detections_table(load_run(path, cfg.refine), real, model) for name, path in RUNS.items()}
for name, df in tables.items():
    print(f"\n{name}: {len(df)} detections, {df.matched.sum()} matched to pseudo-GT (of {N_PARTS} parts)")
    display(accuracy_table(df, diameter=model.diameter))

### 단계별 비교: 시뮬레이션과 실제

시뮬레이션은 검증 장면 20장(정확한 GT), 실제는 70장(의사 정답)입니다. 열은 왼쪽부터 기능을 하나씩 켠 결과입니다.
- **before**: 깊이 맞춤, 중복 제거, 깊이 검증을 모두 끔
- **anchor + dedup**: 깊이 맞춤과 중복 제거만 켬
- **+ verification**: 현재 설정

표의 용어는 다음과 같습니다.
- **false detections in output**: 출력 중 정답 마스크와 IoU 0.5 이상으로 매칭되지 않은 검출입니다.
- **confident인데 틀림**: confident로 표시됐지만 정답과 1mm 이상 다른 pose입니다. 로봇이 잘못 집을 수 있는 위험한 경우입니다.
- **confident 오검출**: 정답과 매칭되지 않았는데 confident로 표시된 검출입니다.

In [ ]:
def summary(df, rejected=None):
    m = df[df.matched]
    c = m[m.confident]
    n_frames = df.view.nunique()
    out = {"outputs per frame": round(len(df) / n_frames, 1),
           "matched": len(m),
           "false detections in output": int((~df.matched).sum()),
           "ICP accepted (share of matched)": f"{m.icp_refined.mean():.0%}",
           "confident per frame": round(df.confident.sum() / n_frames, 1),
           "confident (share of matched)": f"{m.confident.mean():.0%}",
           "confident: ADD-S < 1 mm": f"{(c.final_add_s < 1).mean():.1%}",
           "confident but wrong (>= 1 mm)": int((c.final_add_s >= 1).sum()),
           "confident false detections": int((df.confident & ~df.matched).sum()),
           "confident: centre err median (mm)": round(c.final_centre_mm.median(), 2),
           "confident: tilt median (deg)": round(c.final_tilt_deg.median(), 2)}
    if rejected is not None:
        rj = [d for ds in rejected.values() for d in ds]
        out["rejected by verification (of them real parts)"] = f"{len(rj)} ({sum(d['gt_inst_id'] is not None for d in rj)})"
    return out

def have(path):
    return bool(list(Path(path).glob("*_*.json")))

compare = {}
for name in RUNS:
    cols = {}
    if sim:
        for label, runs in (("sim, before", SIM_BEFORE), ("sim, anchor + dedup", SIM_ANCHOR), ("sim, + verification", SIM_RUNS)):
            if have(runs[name]):
                cols[label] = summary(detections_table(load_run(runs[name], cfg.refine), sim, model), load_rejected(runs[name]))
    cols["real, before"] = summary(detections_table(load_run(RUNS_BEFORE[name], cfg.refine), real, model))
    if name in RUNS_NOREJECT:
        cols["real, anchor + dedup"] = summary(detections_table(load_run(RUNS_NOREJECT[name], cfg.refine), real, model))
    cols["real, + verification"] = summary(tables[name], load_rejected(RUNS[name]))
    compare[name] = pd.DataFrame(cols).fillna("")
    print(name); display(compare[name])

**읽는 법:**
- **깊이 맞춤:** 실제 데이터에서 confident가 장당 3.0개에서 5.4개(장당 부품 5.6개 중)로 늘었습니다. confident pose의 98.9%가 의사 정답과 1mm 이내입니다.
  1mm를 넘은 4개는 1.03~1.26mm로, 의사 정답의 불확실성(약 0.5mm)에서 크게 벗어나지 않습니다.
- **깊이 검증:** 깊이 맞춤과 중복 제거만으로는 실제 데이터에 오검출이 233개(장당 3.3개) 남고, 그중 13개는 confident였습니다.
  깊이 검증을 켜면 출력에서 정답과 매칭되지 않는 것이 **3개**로 줄고, 이 3개도 실제 부품입니다(아래 표). confident pose 수와 정확도는 그대로입니다.
- **시뮬레이션(가득 찬 bin):** 오검출이 60개에서 4개로 줄었습니다. 남은 4개도 실제 부품 위의 부분 마스크이고, pose 오차는 3개가 0.1mm, 가려진 1개가 2.5mm입니다.
  검증에서 빠진 실제 부품 495개는 모두 원래 confident가 아니던 것입니다. 그래서 confident는 장당 72.6개에서 72.5개로 거의 변하지 않습니다.
  쌓인 bin에서 **깊이 맞춤 때문에 생긴 "confident인데 틀린 pose"(4개 → 24개)는 검증으로 줄지 않습니다.** 이 오류는 평면 안의 회전 오류라 depth에 잘 맞기 때문입니다(§9).

### 오검출 제거: 깊이 검증

detector는 시뮬레이션 이미지로만 학습해서 실제 이미지의 bin 테두리, bin 바깥 상자, 타공 바닥 일부, 부품의 일부분에도 마스크를 냅니다.
이런 마스크는 detector score로는 걸러지지 않습니다(score 1.00인 오검출도 있음). 그래서 **각 검출의 최종 pose를 실측 depth로 확인하고, 확인되지 않으면 출력에서 뺍니다**
(`bin6d/refine.py`의 `reject_unverified`, 설정 `refine.reject_unverified`). 걸러진 검출은 `<run>/rejected/`에 이유와 함께 저장됩니다.

| 검사 | 기준 | 거르는 것 |
|---|---|---|
| depth와 ICP | ICP가 수렴하고 보정량이 허용 범위 안 | depth가 없는 bin 바깥(`no_depth`), CAD로 설명되지 않는 표면(`icp_rejected`) |
| fit 품질 | inlier 비율 ≥ 0.5, fit 오차 ≤ 이 프레임 중앙값의 1.5배 | 부품의 일부에만 맞은 fit, bin 바깥 상자 위의 평면(`poor_fit`) |
| 마스크 설명 | 최종 pose의 CAD 실루엣이 마스크의 ≥ 80%를 덮음 | 부품 조각에 맞춘 pose, 바닥이나 이웃까지 덮은 마스크(`mask_not_explained`) |
| 중복 | 마스크의 50% 이상이 더 큰 출력 마스크 안에 있지 않음 | 같은 부품의 부분 마스크(`inside_other_mask`) |

기준을 정할 때 본 분포는 다음과 같습니다(이전 20장과 시뮬레이션 5장).
- 실제 부품은 마스크 설명 비율이 0.91 이상(시뮬레이션은 99%가 0.87 이상)이고, inlier 비율이 0.88 이상(시뮬레이션 0.65 이상)입니다.
- 오검출은 이 값들이 대부분 0~0.6입니다.
- 검증은 confidence 판정보다 느슨합니다. 검증을 통과한 검출 중에서 confidence 판정이 다시 등급을 매깁니다.

In [ ]:
def fp_kind(dets, objs, matched_ids):
    """Detections not matched to pseudo-GT by mask IoU >= 0.5. A pose within 2 mm of a part that no other output
    covers is that part with a partial mask (correct pose, IoU < 0.5): not a false detection for picking."""
    rows = []
    for d in dets:
        if d["gt_inst_id"] is None:
            dist = np.linalg.norm(objs["t"] - np.asarray(d["t"]), axis=1)
            k = int(dist.argmin())
            if dist[k] < 2 and int(objs["inst_id"][k]) not in matched_ids:
                kind = "real part, partial mask (pose correct)"
            elif np.linalg.norm(objs["t"][k, :2] - np.asarray(d["t"])[:2]) < 40:
                kind = "false: duplicate / partial on a part"
            else:
                kind = "false: no part nearby (rim / outside bin)"
            rows.append({"kind": kind})
    return rows

rows = []
for label, path in (("anchor + dedup (no verification)", RUNS_NOREJECT[DET]), ("+ verification: output", RUNS[DET])):
    for (s, v), dets in load_run(path).items():
        ids = {d["gt_inst_id"] for d in dets if d["gt_inst_id"] is not None}
        for r in fp_kind(dets, real.frame_meta(ridx[(s, v)])["objects_all"], ids):
            rows.append({"run": label, "subset": subset(v), **r})
fp = pd.DataFrame(rows)
print("outputs not matched to pseudo-GT by mask IoU (detector masks)")
display(fp.groupby(["run", "kind", "subset"]).size().unstack(fill_value=0).assign(total=lambda d: d.sum(1)))

# what verification removed: false detections and real parts, by reason
rej = []
for (s, v), dets in load_rejected(RUNS[DET]).items():
    for d in dets:
        rej.append({"subset": subset(v), "reason": d["reject"], "real part": d["gt_inst_id"] is not None,
                    "ADD-S if real (mm)": d.get("add_s_refined")})
rej = pd.DataFrame(rej)
print("rejected by verification (detector masks)")
display(rej.groupby(["reason", "real part"]).size().unstack(fill_value=0)
        .rename(columns={False: "false detection", True: "real part (lost)"}))
lost = rej[rej["real part"]]
print(f"real parts rejected: {len(lost)}, their pose error (ADD-S, mm): {sorted(lost['ADD-S if real (mm)'].round(1).tolist())}")

# recall: every pseudo-GT part either output (verified), rejected, or never detected
df = tables[DET]
out_ids = {(v, d["gt_inst_id"]) for (s, v), ds in load_run(RUNS[DET]).items() for d in ds if d["gt_inst_id"] is not None}
rows = []
for v, n in PARTS.items():
    found = sum(1 for (vv, _) in out_ids if vv == v)
    rows.append({"subset": subset(v), "scene": "overlapping" if v in OVERLAP else "separated", "parts": n,
                 "output (verified)": found,
                 "false detections": int(((df.view == v) & ~df.matched).sum())})
rec = pd.DataFrame(rows).groupby(["subset", "scene"]).sum()
rec["parts output"] = (rec["output (verified)"] / rec["parts"]).map("{:.0%}".format)
display(rec)

**결과:**
- **오검출(정답과 매칭 안 됨):** 실제 70장에서 233개(검증 전, 기준을 보지 않은 50장에서 180개) → **3개**. 이 3개는 모두 pose가 정답과 0.3~0.4mm인 실제 부품입니다.
  마스크가 부품의 절반쯤만 덮어 IoU가 0.5 미만이라 매칭이 안 됐을 뿐입니다. 이런 부품을 다른 출력이 대신 잡은 경우는 없습니다. 즉 **집으면 안 되는 출력은 0개**입니다.
- **잃은 실제 부품:** 7개(390개 중)이고, 모두 pose가 3~7.6mm 틀린 것입니다. 어차피 쓸 수 없는 pose라서 출력에서 빠지는 편이 맞습니다.
- **기준을 정할 때 보지 않은 50장도 같은 결과입니다.** 진짜 오검출은 0개이고, 출력된 부품 비율도 93%로 처음 20장(91~94%)과 같습니다. 기준이 20장에 맞춰진 것이 아닙니다.
- **출력되지 않은 부품(약 6%)은 대부분 detector가 처음부터 놓친 것입니다**(§4 끝의 자세별 표). 검증으로 생긴 손실이 아닙니다.

## 4. 왜 깊이 맞춤이 필요한가: 네트워크의 깊이(z) 오차

**구조적인 원인:** 네트워크에 들어가는 depth는 **부품 표면 중앙값을 뺀 상대값**입니다(`bin6d/pose_data.py`의 `assemble_input`). 그래서 네트워크는 표면 모양만 알고,
부품이 몇 mm 떨어져 있는지는 모릅니다. 깊이는 **부품이 crop 안에서 얼마나 크게 보이는지**로만 추정합니다(GDRN SITE: `t_z = site_z · f · 지름 / crop 크기`).
부품이 약 100px 크기라서, 크기를 1%(1px)만 잘못 읽어도 700mm 거리에서 약 7mm 오차가 납니다. ICP는 보정량이 15mm를 넘으면 결과를 거부합니다.

In [ ]:
def depth_errors(run, frames):
    idx = frame_index(frames)
    rows = []
    for (s, v), dets in run.items():
        o = frames.frame_meta(idx[(s, v)])["objects_all"]
        g = {int(i): k for k, i in enumerate(o["inst_id"])}
        for d in dets:
            if d["gt_inst_id"] is None:
                continue
            tg = o["t"][g[d["gt_inst_id"]]]
            row = {"network dz (mm)": d["t_rgbd"][2] - tg[2], "ICP accepted": d["icp_refined"]}
            if "t_anchored" in d:
                row["anchored dz (mm)"] = d["t_anchored"][2] - tg[2]
            rows.append(row)
    return pd.DataFrame(rows)

def dz_summary(e, col):
    s = e[col]
    return {"median dz (mm, + = too far)": round(s.median(), 1), "too far": f"{(s > 0).mean():.0%}",
            "|dz| median (mm)": round(s.abs().median(), 1), "|dz| > 15 mm (ICP limit)": f"{(s.abs() > 15).mean():.0%}"}

e_real = depth_errors(load_run(RUNS[GTM], cfg.refine), real)
e_before = depth_errors(load_run(RUNS_BEFORE[GTM], cfg.refine), real)
cols = {}
if sim and have(SIM_RUNS[GTM]):
    e_sim = depth_errors(load_run(SIM_RUNS[GTM], cfg.refine), sim)
    cols["sim: network"] = dz_summary(e_sim, "network dz (mm)")
    cols["sim: after anchor"] = dz_summary(e_sim, "anchored dz (mm)")
cols["real: network"] = dz_summary(e_real, "network dz (mm)")
cols["real: after anchor"] = dz_summary(e_real, "anchored dz (mm)")
print("depth error vs (pseudo-)GT, GT masks")
display(pd.DataFrame(cols))
g = e_before.groupby("ICP accepted")["network dz (mm)"]
print("real, before anchoring: network |dz| by ICP outcome")
display(pd.DataFrame({"detections": g.size(), "|dz| median (mm)": g.apply(lambda s: s.abs().median()).round(1),
                      "|dz| > 15 mm": g.apply(lambda s: f"{(s.abs() > 15).mean():.0%}")}))

fig = go.Figure()
for col, c, label in (("network dz (mm)", S2, "network"), ("anchored dz (mm)", S3, "after depth anchor")):
    fig.add_trace(go.Histogram(x=e_real[col], name=label, marker_color=c, xbins=dict(start=-10, end=40, size=1),
                               opacity=0.8))
fig.add_vline(x=15, line_dash="dash", line_color="#52514e", annotation_text="ICP limit 15 mm")
style(fig, "Depth error before ICP (real data, GT masks)").update_layout(barmode="overlay")
fig.update_xaxes(title="dz = predicted - pseudo-GT (mm), + = too far").update_yaxes(title="detections")
fig.show()

- **시뮬레이션에서는 깊이 오차가 무작위입니다**(멀게/가깝게가 반반). 실제에서는 **95%의 부품을 약 1% 작게, 즉 약 7mm 멀게** 읽습니다.
  무엇이 부품을 작게 보이게 하는지(밝기, 테두리, 타공 바닥 위의 단독 부품 등)는 확정하지 못했습니다. 카메라 거리는 같아서(§1) 원인이 아닙니다.
- 깊이 맞춤 전에 ICP가 거부된 검출은 대부분(74%) 깊이 오차가 15mm를 넘었습니다. 회전과 평면 위치는 ICP가 고칠 수 있는 범위였습니다.
- **깊이 맞춤**(`bin6d/refine.py`의 `depth_anchor`)은 pose를 카메라 광선 방향으로 밀어서, CAD의 카메라 쪽 표면이 마스크 안의 실측 depth와 맞게 합니다.
  2D 위치와 회전은 그대로이고, 재학습이 필요 없습니다. 맞춘 뒤의 깊이 오차는 중앙값 0.2mm이고, 15mm를 넘는 것은 없습니다.

### confidence 기준 비교 (실제 데이터)

In [ ]:
def variant(path, min_fit, ratio, cap):
    rc = Cfg(dict(cfg.refine)); rc.confidence_min_fitness, rc.confidence_rmse_ratio, rc.confidence_max_rmse_mm = min_fit, ratio, cap
    d = detections_table(load_run(path, rc), real, model)
    c = d[d.confident & d.matched]
    return {"confident": int(d.confident.sum()), "per frame": round(d.confident.sum() / d.view.nunique(), 1),
            "confident false detections": int((d.confident & ~d.matched).sum()),
            "ADD-S < 1 mm": f"{(c.final_add_s < 1).mean():.1%}" if len(c) else "-", "wrong (>= 1 mm)": int((c.final_add_s >= 1).sum())}
for name, path in RUNS.items():
    print(name)
    display(pd.DataFrame({"fixed 0.8 / 1.0 mm": variant(path, 0.8, 1e9, 1.0), "fixed 0.9 / 0.5 mm": variant(path, 0.9, 1e9, 0.5),
                          "relative 0.9 / 1.3x median (cap 1.5 mm)": variant(path, 0.9, 1.3, 1.5)}).T)
acc = tables[DET][tables[DET].icp_refined]
print(f"ICP-accepted fits on real data: fitness median {acc.icp_fitness.median():.3f}, fit error (RMSE) median {acc.icp_rmse.median():.2f} mm")

실제 depth의 fit 오차는 약 0.47mm로, 시뮬레이션의 0.1mm 미만보다 훨씬 큽니다. 고정 기준 0.5mm는 이 값 바로 위에 있습니다.
그래서 70장에서도 맞는 pose 7개(detector 마스크 기준)를 잃고, depth가 조금만 더 거친 촬영이면 훨씬 많이 잃게 됩니다.
프레임마다 기준을 맞추는 상대 기준(현재 설정)을 유지하는 게 안전합니다.

### 프레임별 결과와 겹친 장면

In [ ]:
for name in RUNS:
    df = tables[name]
    m = df[df.matched]
    print(name)
    g = m.groupby([m.view.map(subset), np.where(m.view.isin(OVERLAP), "overlapping", "separated")])
    display(pd.DataFrame({"matched": g.size(), "confident": g.confident.mean().map("{:.0%}".format),
                          "confident ADD-S < 1 mm": g.apply(lambda s: f"{(s[s.confident].final_add_s < 1).mean():.1%}"),
                          "confident wrong (>= 1 mm)": g.apply(lambda s: int((s[s.confident].final_add_s >= 1).sum()))}))
    pf = per_frame_table(df, real)
    bad = pf[(pf["confident correct (<1 mm)"] < pf["parts in bin"]) | (pf["confident wrong"] > 0)]
    print(f"frames where not every part is confident and correct: {len(bad)} of {len(pf)}")
    display(bad.T)

### 부품 자세별 결과

detector가 놓친 부품은 거의 모두 한쪽 자세입니다. CAD z축이 카메라 반대쪽을 향한 자세이고, 개수로 보면 README의 "면이 위" 192개와 같습니다.


In [ ]:
# every pseudo-GT part: found by the detector (matched, before verification) / output after verification / confident
rows = []
nore, ver = load_run(RUNS_NOREJECT[DET], cfg.refine), load_run(RUNS[DET], cfg.refine)
for v in range(len(real)):
    o = real.frame_meta(ridx[(SCENE, v)])["objects_all"]
    found = {d["gt_inst_id"] for d in nore[(SCENE, v)] if d["gt_inst_id"] is not None}
    out = {d["gt_inst_id"]: d for d in ver[(SCENE, v)] if d["gt_inst_id"] is not None}
    for k, iid in enumerate(o["inst_id"]):
        d = out.get(int(iid))
        rows.append({"pose": "CAD z away from camera" if o["R"][k][2, 2] > 0 else "CAD z towards camera",
                     "subset": subset(v), "found by detector": int(iid) in found, "output (verified)": d is not None,
                     "confident": bool(d and d["confident"]),
                     "confident < 1 mm": bool(d and d["confident"] and d["add_s_refined"] < 1)})
parts = pd.DataFrame(rows)
g = parts.groupby("pose")
display(pd.concat([g.size().rename("parts"), g[["found by detector", "output (verified)", "confident", "confident < 1 mm"]]
                   .mean().map("{:.1%}".format)], axis=1))

- **CAD z축이 카메라 반대쪽인 자세에서 detector가 10%를 놓칩니다**(반대 자세는 0.5%). 놓친 21개 중 20개가 이 자세입니다.
  아래 34번 장면의 오른쪽 두 부품이 예입니다. detector가 부품 일부만 덮는 마스크를 내거나, depth가 적은 마스크를 내서 검증에서 빠집니다.
  시뮬레이션과 실제 사이에서 **이 면의 겉모습이 다르다**는 뜻이고, 실제 이미지로 detector를 fine-tuning할 때 가장 먼저 개선될 부분입니다.
- 일단 출력된 부품의 pose 정확도는 두 자세가 같습니다.

## 5. 시각 확인

2D 오버레이: 각 검출의 3D 박스를 투영해 그렸습니다.
- **초록** = confident이고 의사 정답과 1mm 이내
- **빨강** = confident인데 틀림
- **주황** = confident 아님
- **얇은 흰색** = pose가 맞지 않은 곳의 의사 정답
- **자홍** = detector가 놓친 부품

confident인 부품에는 xyz 축을 그립니다(x 빨강, y 초록, z 파랑). 정답과 매칭되지 않은 confident 검출은 초록 박스로 표시됩니다(정답 비교 불가).

- **34번:** 검증 전(위)에는 bin 바깥 상자 위에 오검출이 있습니다. 검증 후(아래)에는 사라집니다. 오른쪽의 두 부품은 위에서 말한 자세라 detector가 제대로 잡지 못합니다.
- **59번:** 부품 10개가 겹친 장면입니다. 검증 전에는 bin 테두리, bin 바깥, 부품 위의 중복 마스크에서 오검출 8개가 나옵니다. 검증 후에는 0개이고, 부품 10개는 모두 confident이며 의사 정답과 1mm 이내입니다.

In [ ]:
def overlay(view, path, label, show="all", width=1000, crop=None):
    f = load_frame(real, ridx[(SCENE, view)])
    dets = load_run(path, cfg.refine)[(SCENE, view)]
    gt = gt_overlay_sets(f["objects_all"], dets, cfg.pose.min_visib_px)
    dets, gt = select(dets, gt, show)
    img = render_2d(f["rgb"], f["K"], dets, model, gt, labels="all", title=f"view {view} ({label}): {show}")
    img = img.crop(crop) if crop else img
    display(img.resize((width, int(width * img.height / img.width))))
overlay(34, RUNS_NOREJECT[DET], "detector, without verification")
overlay(34, RUNS[DET], "detector, + verification")
overlay(59, RUNS_NOREJECT[DET], "detector, without verification")
overlay(59, RUNS[DET], "detector, + verification")

**대화형 3D** (드래그로 회전, 스크롤로 확대, 박스에 마우스를 올리면 fitness, fit 오차, ADD-S 표시). 부품 10개가 겹친 59번 장면입니다.

In [ ]:
v = 59
f = load_frame(real, ridx[(SCENE, v)])
dets = load_run(RUNS[DET], cfg.refine)[(SCENE, v)]
gt = gt_overlay_sets(f["objects_all"], dets, cfg.pose.min_visib_px)
plotly_scene(f["rgb"], f["xyz"], f["valid"], dets, model, gt, max_points=40000,
             title=f"view {v}: detections over the real point cloud, camera frame (mm)").show()

## 6. detector fine-tuning: 시뮬레이션 + 실제 데이터

§3~§5에서 남은 손실은 대부분 detector에서 생겼습니다. 한쪽 자세를 놓치고, 실제 배경에 오검출을 많이 냈습니다.
그래서 `finetuning_detector.py`로 시뮬레이션 학습 detector를 **시뮬레이션과 실제 데이터를 섞어** fine-tuning했습니다(설정 `detector.finetune`).

| | |
|---|---|
| 시작점 | `runs/detector/best.pt` (시뮬레이션만으로 학습) |
| 학습 데이터 | 시뮬레이션 900장(scene 000000~000008) + 실제 47장. 배치의 절반이 실제 이미지가 되도록 실제 이미지를 반복해서 뽑음 |
| 실제 평가용 | **23장**(1, 4, 7, …, 67번, `runs/detector_ft/real_split.json`). 학습에 쓰지 않음 |
| 학습 | 12 epoch × 150 iteration, 학습률 0.002, 약 50분(GB10). 실제 평가용 장면과 시뮬레이션 검증 장면의 평균 F1이 가장 좋은 epoch 9를 선택 |
| detector 단독 평가 (실제 평가용 23장) | precision 0.56 → 1.00, recall 0.94 → 1.00 |
| detector 단독 평가 (시뮬레이션 검증 장면) | F1 0.906 → 0.906 |

아래는 detector만이 아니라 **전체 파이프라인**(깊이 맞춤, ICP, 깊이 검증 포함)으로 비교한 결과입니다. 공정한 비교는 실제 평가용 23장입니다.


In [ ]:
from bin6d.report import _tilt_deg as _tilt

FT_SPLIT = json.load(open("runs/detector_ft/real_split.json"))
FT_VAL = sorted(FT_SPLIT["real_val_views"])

def pipeline_eval(verified, noreject, views):
    """Detector + full pipeline on the given real views: parts found, output after verification, false detections."""
    out, nore = load_run(verified, cfg.refine), load_run(noreject, cfg.refine)
    r = dict(parts=0, found=0, output=0, ok=0, wrong=0, fp_before=0, fp_after=0, partial=0, away=0, away_found=0, away_out=0)
    for v in views:
        o = real.frame_meta(ridx[(SCENE, v)])["objects_all"]
        found = {d["gt_inst_id"] for d in nore[(SCENE, v)] if d["gt_inst_id"] is not None}
        outd = {d["gt_inst_id"]: d for d in out[(SCENE, v)] if d["gt_inst_id"] is not None}
        r["parts"] += len(o["t"]); r["found"] += len(found); r["output"] += len(outd)
        for d in outd.values():
            if d["confident"]:
                r["ok" if d["add_s_refined"] < 1 else "wrong"] += 1
        for k, iid in enumerate(o["inst_id"]):
            if o["R"][k][2, 2] > 0:
                r["away"] += 1; r["away_found"] += int(iid) in found; r["away_out"] += int(iid) in outd
        r["fp_before"] += sum(d["gt_inst_id"] is None for d in nore[(SCENE, v)])
        for kind in fp_kind(out[(SCENE, v)], o, set(outd)):
            r["partial" if kind["kind"].startswith("real part") else "fp_after"] += 1
    n = r["parts"]
    return {"parts": n, "found by detector": f"{r['found']} ({r['found'] / n:.1%})",
            "output (verified)": f"{r['output'] + r['partial']} ({(r['output'] + r['partial']) / n:.1%})",
            "confident < 1 mm": r["ok"], "confident >= 1 mm": r["wrong"],
            "false detections before verification": r["fp_before"], "false detections in output": r["fp_after"],
            "pose with CAD z away from camera: found": f"{r['away_found']}/{r['away']}",
            "pose with CAD z away from camera: output": f"{r['away_out']}/{r['away']}"}

for title, views in ((f"held-out real views ({len(FT_VAL)}, not used in fine-tuning)", FT_VAL),
                     ("  of them overlapping", [v for v in FT_VAL if v in OVERLAP]),
                     (f"training real views ({len(real) - len(FT_VAL)}, seen in fine-tuning)",
                      [v for v in range(len(real)) if v not in set(FT_VAL)])):
    print(title)
    display(pd.DataFrame({"sim-trained detector": pipeline_eval(RUNS[DET], RUNS_NOREJECT[DET], views),
                          "fine-tuned detector": pipeline_eval(RUNS_FT[DET], RUNS_FT_NOREJECT[DET], views)}))

# held-out parts still not in the output: why
rej = load_rejected(RUNS_FT[DET])
outs = load_run(RUNS_FT[DET], cfg.refine)
for v in FT_VAL:
    o = real.frame_meta(ridx[(SCENE, v)])["objects_all"]
    got = {d["gt_inst_id"] for d in outs[(SCENE, v)]}
    for k, iid in enumerate(o["inst_id"]):
        if int(iid) in got:
            continue
        r = [d for d in rej.get((SCENE, v), []) if d["gt_inst_id"] == int(iid)]
        if r:
            d = r[0]
            print(f"view {v}: part {iid} detected but rejected ({d['reject']}); network tilt error "
                  f"{_tilt(d['R_rgbd'], o['R'][k]):.1f} deg, final ADD-S {d['add_s_refined']:.1f} mm")
        else:
            print(f"view {v}: part {iid} not detected")

if sim and have(SIM_RUNS_FT[DET]):
    print("sim val scene (full bins), current pipeline")
    display(pd.DataFrame({"sim-trained detector": summary(detections_table(load_run(SIM_RUNS[DET], cfg.refine), sim, model), load_rejected(SIM_RUNS[DET])),
                          "fine-tuned detector": summary(detections_table(load_run(SIM_RUNS_FT[DET], cfg.refine), sim, model), load_rejected(SIM_RUNS_FT[DET]))}))

**결과 (실제 평가용 23장, 부품 123개, 학습에 쓰지 않음):**
- **놓친 부품이 사라졌습니다.** detector가 찾은 부품이 94.3%에서 **100%**로 늘었습니다.
  놓치던 자세(CAD z축이 카메라 반대쪽)도 51/58에서 **58/58**이 됐습니다. 이 면이 시뮬레이션과 실제에서 다르게 보였다는 해석과 맞습니다.
- **오검출이 detector 단계에서 사라졌습니다.** 깊이 검증 전 오검출이 76개에서 **0개**로 줄었습니다. 이제 깊이 검증은 안전장치 역할만 합니다.
- **출력된 부품은 93.5%에서 99.2%로 늘었고,** 출력된 부품은 모두 confident이며 의사 정답과 1mm 이내입니다.
  출력되지 않은 1개(4번 장면)는 detector가 찾았지만 **GDRN의 기울기 오차가 23°**라 ICP가 수렴하지 못하고 검증에서 빠진 것입니다. detector가 아니라 pose 네트워크의 약점입니다.
- **쌓인 bin(시뮬레이션)은 거의 그대로입니다.** 매칭된 검출이 1% 줄고(1453 → 1438), confident 정확도는 98.3%에서 98.4%로 같습니다. 시뮬레이션 데이터를 섞어 학습한 덕분으로 보입니다.

**주의:** 실제 평가용 23장은 학습용 47장과 **같은 bin, 조명, 배경, 촬영 세션**입니다. 그래서 100%는 낙관적인 수치입니다.
다른 날의 촬영, 다른 조명, 특히 실제로 가득 찬 bin에서 다시 확인해야 합니다.

34번 장면: 위는 시뮬레이션 학습 detector로, 오른쪽 두 부품을 놓쳤습니다. 아래는 fine-tuning한 detector로, 5개 모두 confident이고 0.4~0.7mm입니다.

In [ ]:
overlay(34, RUNS[DET], "sim-trained detector")
overlay(34, RUNS_FT[DET], "fine-tuned detector")

## 7. 가득 찬 실제 bin: `data/real_capture_bop` (정답 없음)

지금까지의 실제 데이터는 장당 부품이 2~10개였습니다. 실제 사용 조건인 **부품이 가득 쌓인 bin**을 Photoneo로 10장 찍은 데이터에 전체 파이프라인을 적용합니다.

| | Photoneo 70장 (§1~§6) | `real_capture_bop` |
|---|---|---|
| 장면 | 장당 부품 2~10개 | **가득 찬 bin**(보이는 부품 100개 이상) |
| 정답 | FoundationPose 의사 정답 | **없음** (`scene_gt.json`이 비어 있음) |
| 카메라 공간 | 1236×1032, fx 1412 (학습과 같음) | **1952×1460, fx 2070** (컬러 카메라 해상도) |
| 부품까지 거리 | 약 700mm | 약 590mm |
| 렌즈 왜곡 보정 | 됨 | **안 됨** (`export_signature.json`의 `undistort: false`) |

- **부품이 이미지에서 약 1.75배 크게 보입니다**(지름 약 195px, 학습은 약 112px). 그래서 detector 입력을 줄이는 옵션 `infer.det_scale`을 추가했습니다.
  0.55로 하면 이미지를 줄여서 검출하고, 마스크는 원래 해상도로 되돌립니다. pose 네트워크는 crop 크기로 정규화하므로 영향이 없습니다.
- **정답이 없어서 정답 대비 정확도는 잴 수 없습니다.** 대신 다음 지표로 평가합니다.
  - 장당 confident pose 수
  - ICP fit 오차
  - CAD 표면과 depth의 모순(violation: CAD가 있어야 할 곳 뒤로 센서가 보는 비율)
  - **depth 설명 지도**: 모든 confident pose의 CAD를 그려서, 쌓인 부품 영역의 depth 중 몇 %를 설명하는지(`bin6d/report.py`의 `explained_depth`)

In [ ]:
from bin6d.report import explained_depth, gt_free_table

CAP_ROOT = "data/real_capture_bop"
cap = BinPoseDataset(CAP_ROOT)
cidx = frame_index(cap)
CSCENE = cap.frames[0].scene
cap_sig = json.load(open(Path(CAP_ROOT) / "export_signature.json"))
cap_cam = cap_sig["camera"]
CAP_RUNS = {("sim-trained", 1.0): "runs/capture_sim_s100", ("sim-trained", 0.55): "runs/capture_sim_s055",
            ("fine-tuned", 1.0): "runs/capture_ft_s100", ("fine-tuned", 0.55): "runs/capture_ft_s055"}
CAP_MAIN = CAP_RUNS[("fine-tuned", 0.55)]                       # current default detector, detector input scaled to the training px size
for (det_name, scale), out in CAP_RUNS.items():
    if RERUN or not list(Path(out).glob("*_*.json")):
        cmd = [sys.executable, "infer.py", "--out", out, "--no-viz", "--set", f"data.root={CAP_ROOT}", "--scenes", CSCENE,
               "--view-stride", "1", "--detector", DET_FT if det_name == "fine-tuned" else DET_SIM,
               "--set", f"infer.det_scale={scale}"]
        print(" ".join(cmd))
        subprocess.run(cmd, check=True, env={**os.environ, "HF_HUB_OFFLINE": "1"})

z_cap = np.median([d["t"][2] for ds in load_run(CAP_MAIN).values() for d in ds])
z_real = float(np.median(np.concatenate([real.frame_meta(i)["objects_all"]["t"][:, 2] for i in range(len(real))])))
px_per_mm = {"Photoneo 70 frames (training camera)": real.frame_meta(0)["K"][0, 0] / z_real,
             "real_capture_bop": cap_cam["fx"] / z_cap}
print(f"{len(cap)} frames, {cap_cam['width']}x{cap_cam['height']}, fx {cap_cam['fx']:.0f}, undistorted: {cap_sig.get('undistort')}, "
      f"no GT: {all(len(cap.frame_meta(i)['objects_all']['t']) == 0 for i in range(len(cap)))}")
print(f"camera -> part distance (median of poses): {z_cap:.0f} mm | part diameter in px: training camera ~"
      f"{model.diameter * px_per_mm['Photoneo 70 frames (training camera)']:.0f}, this capture ~{model.diameter * px_per_mm['real_capture_bop']:.0f} "
      f"-> scale {px_per_mm['Photoneo 70 frames (training camera)'] / px_per_mm['real_capture_bop']:.2f}")
f0 = load_frame(cap, cidx[(CSCENE, 0)])
display(Image.fromarray(f0["rgb"]).resize((900, int(900 * f0["rgb"].shape[0] / f0["rgb"].shape[1]))))

In [ ]:
def cap_summary(path):
    run, rej = load_run(path, cfg.refine), load_rejected(path)
    outs = [d for ds in run.values() for d in ds]
    rj = [d for ds in rej.values() for d in ds]
    nf = len(run)
    share_c, share_a = [], []
    for (s, v), ds in run.items():
        f = load_frame(cap, cidx[(s, v)])
        share_c.append(explained_depth([d for d in ds if d["confident"]], f["depth"], f["K"], model)["share"])
        share_a.append(explained_depth(ds, f["depth"], f["K"], model)["share"])
    reasons = pd.Series([d["reject"] for d in rj]).value_counts()
    return {"detections per frame": round((len(outs) + len(rj)) / nf, 1),
            "output (verified) per frame": round(len(outs) / nf, 1),
            "confident per frame": round(sum(d["confident"] for d in outs) / nf, 1),
            "rejected: icp / poor fit / no depth / mask": "/".join(str(int(reasons.get(k, 0))) for k in
                                                              ("icp_rejected", "poor_fit", "no_depth", "mask_not_explained")),
            "fit error median (mm)": round(float(np.median([d["icp_rmse"] for d in outs])), 3),
            "pile depth explained by confident poses": f"{np.mean(share_c):.1%}",
            "pile depth explained by all outputs": f"{np.mean(share_a):.1%}"}

cap_cmp = pd.DataFrame({f"{d}, det_scale {s}": cap_summary(path) for (d, s), path in CAP_RUNS.items()})
display(cap_cmp)

g = gt_free_table(load_run(CAP_MAIN, cfg.refine), cap, model)
c = g[g.confident]
print(f"current setting ({CAP_MAIN}), {len(c)} confident poses, label-free checks:")
display(c[["anchor_dz_mm", "icp_shift_mm", "icp_rot_deg", "icp_rmse", "surf_violation", "surf_occluded"]]
        .describe(percentiles=[0.05, 0.5, 0.95]).loc[["5%", "50%", "95%", "max"]].round(3))

**읽는 법:**
- **장당 confident pose는 약 70개입니다.** 네 조합(detector 2종 × 입력 배율 2종) 모두 67~72개로 비슷합니다.
  `det_scale 0.55`가 ICP 거부를 줄여서(약 510 → 365) confident가 장당 2~3개 더 많습니다.
- **fine-tuning한 detector가 여기서는 더 낫지 않습니다**(69.9개 vs 71.8개). fine-tuning에 쓴 Photoneo 70장은 부품이 어둡게 찍혔는데, 이 촬영은 밝은 금속색이라 오히려 시뮬레이션에 가깝기 때문으로 보입니다.
- **confident pose는 depth와 모순되지 않습니다.**
  - violation 중앙값은 1.2%이고 95%가 6% 이하입니다. CAD가 있어야 할 곳에 빈 공간이 보이는 경우가 거의 없습니다.
  - fit 오차 중앙값은 0.41mm로 Photoneo 70장(0.47mm)과 비슷합니다.
  - occluded가 높은 것(중앙값 52%)은 이웃 부품과 허브에 가려진 CAD 면 때문입니다.
- **네트워크는 여기서 깊이를 약 6% 가깝게 읽습니다**(깊이 맞춤 이동량 중앙값 −35mm). Photoneo 70장에서는 반대로 1% 멀게 읽었습니다.
  카메라 공간과 거리가 학습과 다르기 때문으로 보이며, ICP 전에 깊이 맞춤이 꼭 필요합니다.
- **쌓인 부품 영역 depth의 약 67%를 confident pose가 설명합니다**(검증된 출력 전체로는 75%). 나머지에는 가려진 아래층 부품, 부품 사이 틈, 놓친 위쪽 부품이 섞여 있습니다.

In [ ]:
# acceptance vs distance from the image centre (lens distortion is not corrected in this export)
K_cap = cap.frame_meta(0)["K"]
rows = []
for path in (CAP_MAIN,):
    for ds in list(load_run(path).values()) + list(load_rejected(path).values()):
        for d in ds:
            if d["n_depth_points"] < cfg.refine.min_points:
                continue
            uv = K_cap @ np.asarray(d["t_anchored"])
            rows.append({"r": float(np.hypot(*(uv[:2] / uv[2] - K_cap[:2, 2]))), "output": "reject" not in d,
                         "confident": bool(d.get("confident")), "rmse": d["icp_rmse"] if d["icp_refined"] else np.nan})
rr = pd.DataFrame(rows)
g = rr.groupby(pd.cut(rr.r, [0, 300, 500, 700, 900, 1300]), observed=True)
display(pd.DataFrame({"detections": g.size(), "output": g.output.mean().map("{:.0%}".format),
                      "confident": g.confident.mean().map("{:.0%}".format),
                      "fit error median (mm)": g.rmse.median().round(3)}).rename_axis("distance from image centre (px)"))

def explained_view(path, view, width=900):
    f = load_frame(cap, cidx[(CSCENE, view)])
    ds = [d for d in load_run(path, cfg.refine)[(CSCENE, view)] if d["confident"]]
    e = explained_depth(ds, f["depth"], f["K"], model)
    img = f["rgb"].astype(np.float32) * 0.6
    for m, col in ((e["explained"], (40, 220, 90)), (e["roi"] & ~e["explained"], (255, 40, 40))):
        img[m] = img[m] * 0.4 + np.array(col) * 0.6
    print(f"view {view}: {len(ds)} confident poses, pile depth explained {e['share']:.0%} "
          "(green = explained by a confident pose, red = depth inside the pile no confident pose explains)")
    im = Image.fromarray(img.astype(np.uint8))
    display(im.resize((width, int(width * im.height / im.width))))

explained_view(CAP_MAIN, 0)
explained_view(CAP_MAIN, 9)

- **이미지 가장자리에서 성능이 떨어집니다.** 중심에서 500px 안에서는 검출의 67%가 confident인데, 700~900px에서는 31%로 떨어집니다. 이때 fit 오차도 0.38에서 0.55mm로 커집니다.
  설명 지도에서도 위쪽에 놓였는데 빨간(설명되지 않은) 부품이 왼쪽 위, 오른쪽, 오른쪽 아래 가장자리에 몰려 있습니다.
  **원인은 렌즈 왜곡 보정 없이 내보낸 데이터로 보입니다.** 파이프라인은 depth를 핀홀 K로만 3D로 바꾸므로, 왜곡된 가장자리의 점군이 휘어져 CAD와 맞지 않습니다.
  `camera/undistort_map.npy`는 1236×1032용이라 이 1952×1460 촬영에는 쓸 수 없습니다.
- **정답이 없어서 "confident인데 평면 안 회전이 틀린 pose"는 잴 수 없습니다.** 시뮬레이션의 가득 찬 bin에서 생겼던 오류(§3, 24개)입니다.
  이 오류는 depth와 모순되지 않아서 위 지표로도 잡히지 않습니다.

**다음에 할 일 (가득 찬 bin):**
1. **학습 카메라 설정으로 다시 촬영합니다.** `camera_convert_data.py` 기본값(PrimaryCamera 1236×1032)으로 찍으면 같은 세션의 왜곡 보정 맵이 `camera.json`에 기록됩니다.
   그다음 `--to-bop`으로 보정해서 내보냅니다. 학습과 같은 해상도라 `det_scale`도 필요 없습니다.
2. **가득 찬 bin 몇 장에 의사 정답을 만들어 평면 안 회전 오류를 잽니다.** 아니면 로봇으로 실제로 집어 봅니다.

## 8. pose 네트워크(GDRN) fine-tuning: 시뮬레이션 + 실제 데이터

detector에 이어 pose 네트워크도 `finetuning_pose.py`로 fine-tuning했습니다(설정 `pose.finetune`).

| | |
|---|---|
| 시작점 | `runs/pose/best.pt` (시뮬레이션만으로 학습). 구조, crop, augmentation 설정을 그대로 씀 |
| 학습 데이터 | 시뮬레이션 900장 + 실제 47장. 배치의 절반이 실제 이미지. 실제 정답은 의사 정답 pose를 CAD에 ray casting해서 만듦 |
| 실제 평가용 | detector fine-tuning과 **같은 23장** |
| 학습 | 12 epoch × 150 iteration, 학습률 3e-5 |
| 모델 선택 | 회전 오차 **중앙값**(실제·시뮬레이션 평균)이 가장 낮은 epoch. 시뮬레이션 검증 장면 100장 전체(crop 1600개)로 평가 |

**처음 학습한 선택 기준(p95)은 실패했습니다.** 처음에는 회전 오차 p95(시뮬레이션 20장, crop 320개)로 골랐습니다.
p95가 상위 16개 crop에 좌우되어 흔들려서, 실제 지표가 계속 좋아지던 epoch 11 대신 epoch 1이 뽑혔습니다(`runs/pose_ft_p95`).
그래서 기준을 중앙값으로 바꾸고 시뮬레이션 평가를 100장으로 늘려 다시 학습했습니다(`runs/pose_ft`, epoch 11 선택). 아래 표는 두 결과를 모두 보여 줍니다.

아래 그래프와 표는 GT 마스크로 자른 crop에서 **네트워크만** 평가한 결과입니다(깊이 맞춤과 ICP 전).

In [ ]:
import re
import torch
POSE_FT = {"p95 criterion, sim 20 views (epoch 1)": "runs/pose_ft_p95", "median criterion, sim 100 views": "runs/pose_ft"}
VAL_RE = re.compile(r"== val (real|sim) ep (-?\d+): (.*)")

def pose_log(path):
    rows = []
    for line in (Path(path) / "train.log").read_text().splitlines():
        m = VAL_RE.search(line)
        if m:
            row = {"set": m.group(1), "epoch": int(m.group(2))}
            row.update({k: float(v) for k, v in (kv.split("=") for kv in m.group(3).split())})
            rows.append(row)
    return pd.DataFrame(rows)

log = pose_log(POSE_FT["median criterion, sim 100 views"])
sel_ep = int(torch.load(Path(POSE_FT["median criterion, sim 100 views"]) / "best.pt", map_location="cpu", weights_only=False)["epoch"])
fig = go.Figure()
for (s, col), c in zip([("real", "rot_deg_median"), ("sim", "rot_deg_median"), ("real", "z_mm_median"), ("sim", "z_mm_median")],
                       [S1, S2, S1, S2]):
    d = log[log.set == s]
    fig.add_trace(go.Scatter(x=d.epoch, y=d[col], name=f"{s}: {'rotation median (deg)' if col.startswith('rot') else 'depth error median (mm)'}",
                             line=dict(color=c, dash="solid" if col.startswith("rot") else "dot"), mode="lines+markers"))
fig.add_vline(x=sel_ep, line_dash="dash", line_color="#52514e", annotation_text=f"best.pt (epoch {sel_ep})")
style(fig, "Pose fine-tuning, network only on GT-mask crops (epoch -1 = sim-trained start)").update_xaxes(title="epoch")
fig.show()
display(log[log.epoch.isin([-1, sel_ep])].set_index(["set", "epoch"])[["rot_deg_median", "rot_deg_p95", "rot_deg_max", "z_mm_median", "add_s_0.1d"]].round(3))

In [ ]:
POSE_RUNS = {"sim-trained pose": ("runs/photoneo70_ft", "runs/photoneo70_ft_noreject"),
             "fine-tuned, p95 / epoch 1": ("runs/photoneo70_ft_poseft", "runs/photoneo70_ft_poseft_noreject"),
             "fine-tuned, p95 run / epoch 11": ("runs/photoneo70_ft_poseft_last", "runs/photoneo70_ft_poseft_last_noreject"),
             "fine-tuned, median (current)": ("runs/photoneo70_ft_poseftmed", "runs/photoneo70_ft_poseftmed_noreject")}

def pose_stats(path, frames, views=None):
    run = load_run(path, cfg.refine)
    if views is not None:
        run = {k: v for k, v in run.items() if k[1] in views}
    m = detections_table(run, frames, model)
    m = m[m.matched]
    return {"net tilt median / p95 (deg)": f"{m.net_tilt_deg.median():.2f} / {m.net_tilt_deg.quantile(.95):.1f}",
            "net centre err median (mm)": round(m.net_centre_mm.median(), 2),
            "anchored (pre-ICP) ADD-S < 2 mm": f"{(m.anchor_add_s < 2).mean():.1%}",
            "final ADD-S median (mm)": round(m.final_add_s.median(), 3)}

for title, views in ((f"held-out real views ({len(FT_VAL)})", FT_VAL),
                     (f"training real views ({len(real) - len(FT_VAL)})", [v for v in range(len(real)) if v not in set(FT_VAL)])):
    print(title, "(fine-tuned detector in all columns)")
    display(pd.DataFrame({name: {**pipeline_eval(a, b, views), **pose_stats(a, real, set(views))}
                          for name, (a, b) in POSE_RUNS.items()}).drop(index=["pose with CAD z away from camera: found"]))

In [ ]:
SIM100 = {"sim-trained pose": "runs/eval100_ft", "fine-tuned, p95 / epoch 1": "runs/eval100_ft_poseftp95",
          "fine-tuned, median (current)": "runs/eval100_ft_poseftmed"}
if sim:
    print("sim val scene, all 100 views (fine-tuned detector in all columns)")
    sim_tabs = {name: detections_table(load_run(path, cfg.refine), sim, model) for name, path in SIM100.items()}
    display(pd.DataFrame({name: {**summary(df, load_rejected(SIM100[name])), **pose_stats(SIM100[name], sim)}
                          for name, df in sim_tabs.items()}))
    # is a difference in "confident but wrong" consistent over the scene? count it per block of 20 views
    rows = {}
    for name, df in sim_tabs.items():
        c = df[df.matched & df.confident]
        rows[name] = c.groupby(c.view // 20 * 20).final_add_s.apply(lambda s: int((s >= 1).sum()))
    blk = pd.DataFrame(rows)
    blk.index = [f"views {i}-{i + 19}" for i in blk.index]
    print("confident but wrong (>= 1 mm) per block of 20 views")
    display(blk.T.assign(total=lambda d: d.sum(1)))

**결과:**
- **네트워크 단독 오차는 좋아집니다.** 실제 평가용 장면(GT 마스크 crop)에서 다음과 같이 바뀌었습니다.
  - 회전 중앙값 2.0° → 1.33°, p95 4.7° → 2.9°
  - 깊이 오차 중앙값 7.1mm → 2.0mm. 부품을 1% 멀게 읽던 편향을 대부분 배웠습니다.
- **그런데 최종 결과는 바뀌지 않습니다.** 실제 평가용 23장에서 네 모델 모두 출력 122/123, confident이고 1mm 이내 122개, 최종 ADD-S 중앙값 0.53mm로 같습니다.
  - 깊이 맞춤과 ICP가 네트워크 오차를 이미 보정하고 있었습니다.
  - 최종 오차는 이미 의사 정답의 한계(약 0.5mm, 1~2°) 수준입니다.
  - 출력되지 않은 1개(4번 장면)는 네 모델 모두 ICP에서 거부됩니다.
- **학습에 쓴 47장에서는 2개를 더 살렸지만**(264 → 266), 본 데이터라 근거로 쓰기 어렵습니다.
- **시뮬레이션 100장:** 출력 수(장당 72.5개)와 confident 정확도(98.4~98.6%)는 그대로입니다.
  - "confident인데 1mm 이상 틀린 pose"는 116개 → 108개(이전 fine-tuning) → 103개(현재 fine-tuning)로 조금 줄었습니다.
  - 하지만 20장씩 나눈 구간별로는 방향이 일정하지 않습니다(60~79번 구간은 13개 → 20개로 늘어남).
  - 처음 20장 평가에서 본 23개 → 14개는 **대부분 우연**이었습니다. 이 오류(평면 안 회전)는 fine-tuning으로 거의 줄지 않습니다.

**판단:** `infer.py`의 기본 pose 모델은 시뮬레이션 학습 모델(`runs/pose/best.pt`)로 유지합니다.
fine-tuning 모델은 측정 가능한 범위에서 나빠지는 곳도 없지만, 실제 데이터에서 얻는 이득도 측정되지 않습니다.
또 이 Photoneo 촬영의 깊이 편향을 배웠기 때문에, 카메라 설정이 다른 촬영(§7은 반대로 6% 가깝게 읽음)에서는 불리할 수 있습니다.
쓰려면 `--pose runs/pose_ft/best.pt`를 붙이면 됩니다.

## 9. 결론, 주의점, 다음 단계

**결론: 시뮬레이션으로만 학습한 모델이 실제 Photoneo 70장에서 동작합니다. 깊이 검증을 넣은 뒤로는 출력에 집으면 안 되는 오검출이 없습니다.**
**detector를 시뮬레이션과 실제 데이터로 fine-tuning하면, 학습에 쓰지 않은 실제 23장에서 출력된 부품이 93.5%에서 99.2%로 늘고 detector 오검출도 0개가 됩니다(§6).**
**가득 찬 실제 bin(정답 없음, §7)에서는 장당 약 70개의 confident pose가 나오고 depth와 모순되지 않습니다.** 다만 렌즈 왜곡을 보정하지 않은 촬영이라 가장자리 성능이 떨어집니다.
**pose 네트워크 fine-tuning(§8)은 네트워크 단독 오차(특히 깊이)를 줄이지만, 최종 결과는 바꾸지 않습니다.** 기본 pose 모델은 시뮬레이션 학습 모델로 유지합니다.
**infer.py의 기본 detector는 이제 fine-tuning한 모델(`runs/detector_ft/best.pt`)입니다.** 아래 표는 시뮬레이션만으로 학습한 detector 기준입니다.

| detector 마스크, 실제 70장 (부품 390개) | 처음 | 깊이 맞춤 + 중복 제거 | + 깊이 검증 (현재) |
|---|---|---|---|
| 장당 출력 (장당 부품 5.6개) | 9.0개 | 8.6개 | **5.2개** |
| 출력 중 오검출 | 264개 | 233개 | **0개** (정답 매칭 실패 3개는 pose가 맞은 실제 부품) |
| confident 오검출 | 2개 | 13개 | **0개** |
| confident pose (장당) | 3.0개 | 5.4개 | 5.2개 |
| confident 중 의사 정답과 1mm 이내 | 99.5% | 98.9% | 98.9% (넘은 4개는 1.03~1.26mm) |
| 출력된 부품 | | | 362 + 3 / 390 (94%) |

- **오검출 원인:** detector가 bin 테두리, bin 바깥(상자, 케이블, depth 없는 검은 영역), 부품의 일부분에 마스크를 냅니다. 시뮬레이션 학습 데이터에 없는 배경입니다.
- **해결:** 각 검출의 최종 pose를 실측 depth로 확인해서, CAD가 그 자리의 표면과 마스크를 설명하지 못하면 버립니다(`refine.reject_unverified`). 재학습은 필요 없습니다.
  기준을 정할 때 보지 않은 50장에서도 진짜 오검출이 0개였습니다. 이 단계에서 잃은 실제 부품은 7개이고, 모두 pose가 3mm 이상 틀린 것이었습니다.
- **남은 약점 1 (fine-tuning으로 해결):** 시뮬레이션 학습 detector는 한쪽 자세(CAD z축이 카메라 반대쪽)를 놓쳐서, 이 자세의 출력률이 86%였습니다(반대 자세 99.5%).
  fine-tuning 후 실제 평가용 장면에서 이 자세도 모두 찾습니다(§6).
- **남은 약점 2:** GDRN이 같은 자세에서 가끔 기울기를 13~23° 틀립니다. 이 경우 ICP가 수렴하지 못해 부품이 출력에서 빠집니다(70장에서 7개, fine-tuning 후 평가용 23장에서 1개).

**주의점**
- **정답이 FoundationPose 결과입니다.** "1mm 이내"는 FoundationPose와 일치한다는 뜻이고, 진짜 정확도는 로봇 집기 성공률 같은 독립적인 기준으로 확인해야 합니다.
  depth로 검증되지 않는 방향(옆 방향 위치, 허브 축 둘레 각도)은 두 방법이 같은 쪽으로 틀릴 수도 있습니다.
- **깊이 검증은 "틀린 위치"는 잡지만 "평면 안 회전 오류"는 못 잡습니다.** 가득 찬 bin(시뮬레이션)에서 깊이 맞춤으로 생긴 confident인데 틀린 pose
  (4개 → 24개, 대부분 위치와 기울기는 맞고 평면 안 회전만 어긋남)는 검증 후에도 24개로 그대로입니다. 부품 중심을 흡착으로 집는다면 영향이 작을 수 있습니다.
- **쌓인 bin에서는 출력 수가 줄어듭니다.** 시뮬레이션에서 장당 출력이 100개에서 73개로 줄지만, 빠진 것은 원래 confident가 아니던 검출이라 집을 수 있는 부품 수는 같습니다.
- **실제 장면은 아직 쉽습니다.** 장당 2~10개이고, 가득 찬 bin은 없습니다.
- **depth 단위:** 이 촬영의 depth는 1mm 단위라, 0.1mm 단위 촬영보다 fit 오차가 커집니다(약 0.47mm).

**다음 단계**
0. **가득 찬 bin 재촬영:** 학습 카메라 설정(1236×1032)과 왜곡 보정으로 다시 찍고, 몇 장에 의사 정답을 만들어 평면 안 회전 오류까지 잽니다(§7).
1. **fine-tuning한 detector를 새 촬영으로 확인하기:** 다른 날짜, 다른 조명, 가득 찬 bin을 찍어서 실제 평가용 23장의 결과가 유지되는지 봅니다.
   새 촬영도 의사 정답을 만들어 `detector.finetune.real_root`에 추가하면 같은 스크립트로 다시 fine-tuning할 수 있습니다.
2. **가득 찬 bin을 실제로 촬영해 확인하기:** 실제 사용 조건이고, 시뮬레이션에서 본 "평면 내 회전 오류"가 실제에서도 나타나는지 봐야 합니다.
3. **GDRN의 큰 기울기 오류 줄이기:** GDRN fine-tuning으로는 4번 장면의 1개가 살아나지 않았습니다(§8). 학습 없이 ICP를 여러 기울기 후보에서 시작해 가장 잘 맞는 결과를 고르는 방법이 다음 후보입니다.
   가득 찬 bin의 평면 안 회전 오류도 fine-tuning으로 거의 줄지 않아서, 같은 방식(여러 회전 후보 + 검증)이 필요할 수 있습니다.
4. **로봇 집기 시험:** confident pose를 base 좌표로 바꿔(`p_base = T_base_cam @ p_cam`) 실제로 집어 보는 것이 의사 정답과 무관한 가장 확실한 검증입니다.

## 10. 부품별 6D pose

위 절들은 정확도만 봤습니다. 여기서는 파이프라인이 **부품마다 내는 6D pose 값 자체**를 표로 꺼냅니다(현재 기본 설정: fine-tuning한 detector, 시뮬레이션 학습 pose).

- **좌표계:** 카메라 좌표계(x 오른쪽, y 아래, z 앞), 단위 mm. 물체 원점은 CAD bbox 중심, 물체 z축은 **허브 축**입니다(이 축으로 180° 대칭).
- **pose:** `p_cam = R @ p_model + t`. 표에는 위치 t(x, y, z), 회전(xyz 오일러 각, 쿼터니언), 허브 축 방향을 적습니다.
- **대칭 주의:** 허브 축 둘레 180° 회전은 같은 자세라, 오일러 각과 쿼터니언은 두 값 중 하나입니다. 비교할 때는 **허브 축 방향**과 **평면 안 각도(mod 180°)**를 보세요.
- **허브가 카메라 쪽:** 허브 축이 카메라를 향하는지입니다(§5의 "CAD z축이 카메라 반대쪽" 자세가 False).
- **depth 확인 (support / violation / occluded):** 그 pose의 CAD 보이는 면을 투영해 실측 depth와 비교한 비율입니다.
  support = 2mm 안에서 맞음, violation = depth가 CAD **뒤**(빈 공간이 보임, pose 모순), occluded = **앞에** 다른 물체(덮임).
  박스 그림은 기울어진 부품에서 찌그러져 보여 pose가 맞는지 판단하기 어렵습니다. **CAD 표면 그림**(초록 = 맞음, 파랑 = 덮임, 빨강 = 모순)으로 확인하세요.
- **pick 후보:** confident이면서 **위에 다른 부품이 없는** 것만, 카메라에 가까운(위에 있는) 순서로 고릅니다.
  기준은 `occluded ≤ 0.30`, `support ≥ 0.60`입니다. Photoneo 70장에서 97% 넘게 보이는 부품은 occluded 중앙값 0.17, p95 0.29(허브가 자기 면을 가리는 몫),
  support 최솟값 0.58이었습니다. 위에 부품이 얹힌 부품은 보이는 일부 면으로만 판정되어 덮인 부분의 pose가 검증되지 않고, 집기에도 나쁩니다.
- **confidence score (0~1):** `support × fitness × min(1, 프레임 맞춤 오차 중앙값 / 이 부품의 맞춤 오차)`. CAD 표면이 depth와 넓게 맞고,
  맞춤 오차가 그 프레임의 보통 수준 이하일수록 높습니다. confident가 아니면 0입니다. "맨 위 + confidence 높은 순" 출력에 씁니다.
- **로봇 좌표:** `T_BASE_CAM`(4×4, mm)을 넣으면 base 좌표 열이 생깁니다(이 장비에서 아직 검증되지 않아 기본값은 None).
- 의사 정답이 있는 장면은 정답과의 위치 차이, 허브 축 각도 차이, ADD-S를 함께 적습니다.

In [ ]:
from scipy.spatial.transform import Rotation
from PIL import ImageDraw
from bin6d.report import _tilt_deg, surface_check

POSE_RUN = RUNS_FT[DET]                       # current default: fine-tuned detector + sim-trained pose (§6)
T_BASE_CAM = None                             # 4x4 camera -> robot base (mm). None = camera frame only
PICK_MAX_OCCLUDED, PICK_MIN_SUPPORT = 0.30, 0.60   # "nothing on top": p95 / min of fully visible parts (Photoneo 70)

def pose_rows(dets, objs=None, frame=None):
    """One row per output detection: 6D pose in the camera frame, depth check (if `frame`), pick order,
    base-frame position (if T_BASE_CAM) and pseudo-GT error (if `objs` has GT)."""
    gid = {int(i): k for k, i in enumerate(objs["inst_id"])} if objs is not None else {}
    acc = [d["icp_rmse"] for d in dets if d["icp_refined"]]
    med_rmse = float(np.median(acc)) if acc else np.nan                # frame's typical fit error
    rows = []
    for n, d in enumerate(dets):
        R, t = np.asarray(d["R"]), np.asarray(d["t"])
        rot = Rotation.from_matrix(R)
        rx, ry, rz = rot.as_euler("xyz", degrees=True)
        qx, qy, qz, qw = rot.as_quat()
        hub = R[:, 2]
        a = np.degrees(np.arctan2(R[1, 0], R[0, 0])) % 180          # in-plane angle of the x axis, 2-fold symmetric
        row = {"det": n, "pickable": False, "confident": d["confident"],
               "x (mm)": t[0], "y (mm)": t[1], "z (mm)": t[2],
               "rx (deg)": rx, "ry (deg)": ry, "rz (deg)": rz, "qw": qw, "qx": qx, "qy": qy, "qz": qz,
               "hub axis (cam)": np.round(hub, 3).tolist(), "hub toward camera": bool(hub @ t < 0),
               "tilt to optical axis (deg)": float(np.degrees(np.arccos(min(abs(hub[2]), 1.0)))),
               "in-plane angle mod 180 (deg)": a, "fit error (mm)": d["icp_rmse"], "fitness": d["icp_fitness"]}
        if frame is not None:
            sc = surface_check(R, t, frame["depth"], frame["K"], model)
            row.update({"support": sc["support"], "violation": sc["violation"], "occluded": sc["occluded"]})
            row["pickable"] = bool(d["confident"] and sc["occluded"] <= PICK_MAX_OCCLUDED and sc["support"] >= PICK_MIN_SUPPORT)
            row["conf score"] = (float(sc["support"] * d["icp_fitness"] * min(1.0, med_rmse / max(d["icp_rmse"], 1e-6)))
                                 if d["confident"] else 0.0)
        if T_BASE_CAM is not None:
            p = np.asarray(T_BASE_CAM) @ np.r_[t, 1.0]
            row.update({"x base (mm)": p[0], "y base (mm)": p[1], "z base (mm)": p[2]})
        k = gid.get(d.get("gt_inst_id")) if d.get("gt_inst_id") is not None else None
        if k is not None:
            row.update({"GT centre err (mm)": float(np.linalg.norm(t - objs["t"][k])),
                        "GT hub axis err (deg)": _tilt_deg(R, objs["R"][k]), "GT ADD-S (mm)": d.get("add_s_refined")})
        rows.append(row)
    df = pd.DataFrame(rows).set_index("det")
    ok = df.pickable if frame is not None else df.confident          # without depth: confident only
    order = {n: r + 1 for r, n in enumerate(df[ok].sort_values("z (mm)").index)}
    df.insert(0, "pick order", pd.array([order.get(n) for n in df.index], dtype="Int64"))   # <NA> = not a candidate
    return df

def show_pose_table(df, digits=2):
    display(df.round(digits))

def surface_overlay(f, dets, ids, title="", crop_margin=90, scale=1.6, draw=None):
    """CAD camera-facing surface of dets[ids] over the image, coloured by the measured depth there:
    green = within 2 mm, blue = something in front (covered), red = depth behind the CAD (pose contradiction),
    grey = no depth. Cropped to the drawn parts."""
    K, depth = f["K"], f["depth"]
    img = Image.blend(Image.fromarray(f["rgb"]).convert("RGB"), Image.new("RGB", (f["rgb"].shape[1], f["rgb"].shape[0])), 0.3)
    dr = ImageDraw.Draw(img)
    centres = []
    for n in ids:
        R, t = np.asarray(dets[n]["R"]), np.asarray(dets[n]["t"])
        pts = model.dense_points @ R.T + t
        pts = pts[((model.dense_normals @ R.T) * pts).sum(1) < 0][::3]
        p = pts @ K.T
        u, v = np.round(p[:, 0] / p[:, 2]).astype(int), np.round(p[:, 1] / p[:, 2]).astype(int)
        ok = (u >= 0) & (u < depth.shape[1]) & (v >= 0) & (v < depth.shape[0])
        u, v, z = u[ok], v[ok], pts[ok, 2]
        dm = depth[v, u]
        diff = dm - z
        for uu, vv, dd, mm in zip(u, v, diff, dm):
            dr.point((uu, vv), fill=(90, 90, 90) if mm == 0 else
                     (40, 230, 90) if abs(dd) <= 2 else (255, 40, 40) if dd > 2 else (60, 120, 255))
        c = K @ t
        centres.append(c[:2] / c[2])
        dr.text((c[0] / c[2] + 5, c[1] / c[2] - 6), f"#{n}", fill=(255, 255, 0))
    if draw is not None:
        draw(dr)
    if title:
        dr.text((8, 8), title, fill=(255, 255, 255))
    if centres and crop_margin:
        c = np.array(centres)
        box = (max(int(c[:, 0].min()) - crop_margin, 0), max(int(c[:, 1].min()) - crop_margin, 0),
               min(int(c[:, 0].max()) + crop_margin, img.width), min(int(c[:, 1].max()) + crop_margin, img.height))
        img = img.crop(box)
        if title:
            ImageDraw.Draw(img).text((8, 8), title, fill=(255, 255, 255))
    return img.resize((int(img.width * scale), int(img.height * scale)))

v = 59                                        # 10 touching / overlapping parts
dets = load_run(POSE_RUN, cfg.refine)[(SCENE, v)]
f59 = load_frame(real, ridx[(SCENE, v)])
pt = pose_rows(dets, f59["objects_all"], f59)
print(f"view {v}: {len(pt)} output parts, {int(pt.confident.sum())} confident, {int(pt.pickable.sum())} pickable ({POSE_RUN})")
show_pose_table(pt)
overlay(v, POSE_RUN, "fine-tuned detector: det index = table row")
display(surface_overlay(f59, dets, list(pt.index), f"view {v}: CAD surface vs depth (green ok, blue covered, red contradiction)"))

first = pt[pt["pick order"] == 1].index[0]
T = np.eye(4); T[:3, :3], T[:3, 3] = np.asarray(dets[first]["R"]), np.asarray(dets[first]["t"])
print(f"pick #1 = det {first}: T_cam_obj (mm)\n{np.array2string(T, precision=3, suppress_small=True)}")

### 가득 찬 실제 bin (§7, 정답 없음): confident 6D pose와 pick 후보

표는 이 장면의 **confident pose 전부**를 카메라에 가까운(위에 있는) 순서로 보여 줍니다(`TOP_N = None`, 숫자를 넣으면 그 개수만).
`pick order`는 그중 **위에 다른 부품이 없는 것**(pickable)에만 붙습니다. 그림은 세 장입니다.

1. confident 전부(박스). 번호는 표의 `det`입니다.
2. **높이만으로 고른 상위 `PICK_N`개의 CAD 표면 그림**(이전 방식). 파란 부분이 많은 부품은 위에 다른 부품이 얹힌 것이라 pick 후보에서 빠집니다.
3. **pick 후보 상위 `PICK_N`개**(높이 + 덮이지 않음)의 CAD 표면 그림.

confident가 아닌 출력과 출력되지 않은 부품(많이 가려졌거나 ICP나 깊이 검증을 통과하지 못한 것)은 그리지 않습니다.
이 촬영은 렌즈 왜곡을 보정하지 않았기 때문에(§7) 이미지 가장자리 부품은 그림이 수 픽셀 어긋날 수 있습니다.

In [ ]:
TOP_N = None                                  # rows in the table: None = all confident poses
PICK_N = 10                                   # parts drawn in figures 2 and 3
cv = 0
cdets = load_run(CAP_MAIN, cfg.refine)[(CSCENE, cv)]
fc = load_frame(cap, cidx[(CSCENE, cv)])
ct = pose_rows(cdets, frame=fc)
conf = ct[ct.confident].sort_values("z (mm)")
picks = conf[conf.pickable]
print(f"capture view {cv}: {len(ct)} output parts, {len(conf)} confident, {len(picks)} pickable "
      f"(occluded <= {PICK_MAX_OCCLUDED}, support >= {PICK_MIN_SUPPORT}) ({CAP_MAIN})")
with pd.option_context("display.max_rows", 500):
    show_pose_table(conf.drop(columns=["confident"]).head(TOP_N or len(conf)))

img = render_2d(fc["rgb"], fc["K"], [d if n in set(conf.index) else None for n, d in enumerate(cdets)], model, None,
                labels="all", title=f"capture view {cv}: all {len(conf)} confident")
display(img.resize((1000, int(1000 * img.height / img.width))))
by_height = list(conf.index[:PICK_N])
dropped = [n for n in by_height if not ct.loc[n, "pickable"]]
print(f"top {len(by_height)} by height only: {len(dropped)} covered -> not pick candidates: "
      + ", ".join(f"#{n} (occluded {ct.loc[n, 'occluded']:.2f}, support {ct.loc[n, 'support']:.2f})" for n in dropped))
display(surface_overlay(fc, cdets, by_height, f"top {len(by_height)} confident by height only (previous rule)"))
display(surface_overlay(fc, cdets, list(picks.index[:PICK_N]), f"pick candidates: top {min(PICK_N, len(picks))} by height, not covered"))

### 맨 위 + confidence 높음 + 가운데 구멍 주변이 비어 있음: 10개의 6D pose 출력

그리퍼가 **가운데 구멍에 손가락을 넣고 벌려서(내경 확장)** 잡는다고 보고, 다음 조건을 모두 만족하는 부품 중 confidence score가 높은 순서로 `TOP_K`개를 고릅니다.

1. **confident**이고 **대체로 덮이지 않음**(occluded ≤ `GRASP_MAX_OCCLUDED`). 테두리가 조금 덮인 것은 허용합니다. 집는 데 중요한 구멍 주변은 아래 3, 4번에서 따로 봅니다.
2. **접근 기울기:** 구멍 축과 카메라 광축 사이의 각도가 `MAX_TILT` 이하입니다. 로봇 base 좌표가 없어서 카메라 광축을 수직 접근 방향으로 봅니다.
3. **구멍이 열려 있음:** 구멍 입구에서 손가락 반경(`R_FINGER`) 안, 삽입 깊이(`INSERT_DEPTH`)까지 다른 물체가 없어야 합니다.
   막힌 비율 = 그 안에서 측정된 점 수 ÷ 그 원이 이미지에서 차지하는 픽셀 수입니다(`HOLE_BLOCK_MAX` 이하).
4. **접근 여유:** 그리퍼 몸체 반경(`R_BODY`) 안의 링에서 구멍 입구보다 `RING_MARGIN` 넘게 튀어나온 물체의 면적 비율이 `RING_BLOCK_MAX` 이하입니다.
   허용 비율 5~10%는 여기서 씁니다. 부품의 바깥 테두리가 다른 부품과 겹치는 것은 상관하지 않습니다.
5. 고른 부품끼리는 이미지에서 겹치지 않게 합니다.

**구멍 위치:** CAD에서 가운데 구멍은 관통이고 내경은 ⌀11.2~13.8mm입니다. 허브(외경 ⌀18mm)가 카메라 쪽이면 허브 윗면에서, 판이 카메라 쪽이면 판 면에서 들어갑니다(`approach side`).
**그리퍼 치수는 임시값입니다**(손가락 ⌀10mm, 삽입 8mm, 몸체 ⌀30mm, 기울기 30°). 실제 값으로 바꾸세요.

출력에는 6D pose와 함께 **집는 점**(구멍 입구 중심, `grasp x/y/z`)과 **접근 방향**(구멍 축, 부품 안쪽 방향 단위 벡터)이 들어 있습니다(모두 카메라 좌표계, mm).
그림의 노란 원은 손가락, 하늘색 원은 그리퍼 몸체 범위이고, 번호는 표의 `det`입니다.
이 촬영은 렌즈 왜곡을 보정하지 않아서(§7) 이미지 가장자리에서는 실측 점이 수 mm 어긋날 수 있습니다. 여유값(`RING_MARGIN`)이 이를 일부 흡수합니다.

In [ ]:
from scipy.ndimage import binary_closing, binary_fill_holes

TOP_K = 10
# gripper (provisional values: replace with the real gripper)
R_FINGER = 5.0          # mm, radius of the closed fingers (hole: 11.2-13.8 mm diameter)
INSERT_DEPTH = 8.0      # mm, finger insertion depth
R_BODY = 15.0           # mm, radius of the gripper body / tip that must clear the surroundings
MAX_TILT = 30.0         # deg, hole axis vs camera optical axis (stand-in for the vertical approach)
RING_MARGIN = 3.0       # mm, surroundings may stick up this much above the hole entrance (noise, calibration)
APPROACH_LEN = 60.0     # mm, height above the entrance checked for obstacles
HOLE_BLOCK_MAX = 0.10   # allowed blocked share of the finger disc
RING_BLOCK_MAX = 0.10   # allowed blocked share of the body ring (the 5-10 % tolerance)
GRASP_MAX_OCCLUDED = 0.5
MAX_OVERLAP_SELECTED = 0.02

def grasp_check(det, frame):
    # entrance of the centre hole on the camera side, hole axis, and what the measured cloud has around it
    R, t = np.asarray(det["R"], float), np.asarray(det["t"], float)
    hub = R[:, 2]
    side = 1.0 if hub @ t < 0 else -1.0                              # +1: hub side faces the camera
    z_top = model.xyz_min[2] + model.xyz_size[2] if side > 0 else model.xyz_min[2]
    entry = R @ np.array([0.0, 0.0, z_top]) + t
    up = side * hub                                                  # out of the part, towards the camera
    tilt = float(np.degrees(np.arccos(np.clip(-up[2], -1, 1))))
    K, xyz, valid = frame["K"], frame["xyz"], frame["valid"]
    c = K @ entry
    u0, v0 = c[0] / c[2], c[1] / c[2]
    half = int(1.5 * (R_BODY + 5) * K[0, 0] / entry[2]) + 2
    hh, ww = valid.shape
    ys = slice(max(int(v0) - half, 0), min(int(v0) + half, hh))
    xs = slice(max(int(u0) - half, 0), min(int(u0) + half, ww))
    d = xyz[ys, xs][valid[ys, xs]].astype(float) - entry
    h = d @ up                                                       # height above the entrance plane
    rho = np.linalg.norm(d - h[:, None] * up, axis=1)                # distance from the hole axis
    # blocked AREA share: blocking points / pixels the disc or ring covers in the image (an open hole has
    # almost no depth inside, so a share of the observed points would turn a few edge-noise points into 100 %)
    px_per_mm2 = (K[0, 0] / entry[2]) * (K[1, 1] / entry[2]) * max(abs(up[2]), 0.2)
    hole_px = np.pi * R_FINGER ** 2 * px_per_mm2
    ring_px = np.pi * (R_BODY ** 2 - R_FINGER ** 2) * px_per_mm2
    hole_blocked = min(float(((rho < R_FINGER) & (h > -INSERT_DEPTH)).sum()) / hole_px, 1.0)
    ring_blocked = min(float(((rho >= R_FINGER) & (rho < R_BODY) & (h > RING_MARGIN) & (h < APPROACH_LEN)).sum())
                       / ring_px, 1.0)
    return {"approach side": "hub" if side > 0 else "plate", "approach tilt (deg)": tilt,
            "hole blocked": hole_blocked, "ring blocked": ring_blocked,
            "grasp x (mm)": entry[0], "grasp y (mm)": entry[1], "grasp z (mm)": entry[2],
            "approach (cam)": np.round(-up, 4).tolist()}

def silhouette(det, frame):
    # projected CAD silhouette (all CAD points, closed and filled) in the image
    p = (model.dense_points @ np.asarray(det["R"]).T + np.asarray(det["t"])) @ frame["K"].T
    u, v = np.round(p[:, 0] / p[:, 2]).astype(int), np.round(p[:, 1] / p[:, 2]).astype(int)
    h, w = frame["depth"].shape
    ok = (u >= 0) & (u < w) & (v >= 0) & (v < h)
    m = np.zeros((h, w), bool)
    m[v[ok], u[ok]] = True
    return binary_fill_holes(binary_closing(m, iterations=3))

def why_not(r, detail=True):
    # first failed condition ("" = graspable); detail=False gives just the category
    if not r["confident"]:
        return "not confident"
    for bad, name, val in ((r["occluded"] > GRASP_MAX_OCCLUDED, "covered", f"occluded {r['occluded']:.2f}"),
                           (r["approach tilt (deg)"] > MAX_TILT, "tilted", f"{r['approach tilt (deg)']:.0f} deg"),
                           (r["hole blocked"] > HOLE_BLOCK_MAX, "hole blocked", f"{r['hole blocked']:.0%}"),
                           (r["ring blocked"] > RING_BLOCK_MAX, "gripper ring blocked", f"{r['ring blocked']:.0%}")):
        if bad:
            return f"{name} {val}" if detail else name
    return ""

def top_picks(df, dets, frame, k=TOP_K):
    # graspable parts (conditions 1-4) in confidence order, skipping any that overlaps a part already selected
    if not df.confident.any():
        return df.iloc[:0].assign(rank=[]), df.assign(**{"grasp ok": False})
    g = pd.DataFrame({n: grasp_check(dets[n], frame) for n in df.index[df.confident]}).T
    num = [c for c in g.columns if c not in ("approach side", "approach (cam)")]
    g[num] = g[num].astype(float)
    full = df.join(g)
    full["grasp ok"] = full.apply(lambda r: r["confident"] and why_not(r) == "", axis=1)
    sil, chosen = {}, []
    for n in full[full["grasp ok"]].sort_values("conf score", ascending=False).index:
        sil[n] = silhouette(dets[n], frame)
        a = max(sil[n].sum(), 1)
        if any((sil[n] & sil[m]).sum() / a > MAX_OVERLAP_SELECTED for m in chosen):
            continue
        chosen.append(n)
        if len(chosen) == k:
            break
    best = full.loc[chosen].copy()
    best.insert(0, "rank", range(1, len(best) + 1))
    return best, full

best, full = top_picks(ct, cdets, fc)
conf_full = full[full.confident]
print(f"capture view {cv}: {len(conf_full)} confident -> {int(full['grasp ok'].sum())} graspable "
      f"(on top, tilt <= {MAX_TILT:.0f} deg, hole <= {HOLE_BLOCK_MAX:.0%} / ring <= {RING_BLOCK_MAX:.0%} blocked) "
      f"-> {len(best)} selected ({CAP_MAIN})")
reasons = conf_full.apply(lambda r: why_not(r, detail=False), axis=1)
print("confident but not graspable (first failed condition):",
      ", ".join(f"{k} {v}" for k, v in reasons[reasons != ""].value_counts().items()))
old = list(conf_full[conf_full.pickable].sort_values("conf score", ascending=False).index[:TOP_K])
dropped = [n for n in old if n not in set(best.index)]
if dropped:
    print("in the previous top 10 (on top + confidence) but dropped now: "
          + ", ".join(f"#{n} ({why_not(full.loc[n]) or 'overlaps a selected part'})" for n in dropped))
print()
for n, r in best.iterrows():
    ax = r["approach (cam)"]
    print(f"#{r['rank']:<2d} det {n:<3d} score {r['conf score']:.3f} | t = ({r['x (mm)']:7.1f}, {r['y (mm)']:7.1f}, {r['z (mm)']:6.1f}) mm"
          f" | R xyz = ({r['rx (deg)']:7.1f}, {r['ry (deg)']:6.1f}, {r['rz (deg)']:7.1f}) deg"
          f" | q wxyz = ({r['qw']:.4f}, {r['qx']:.4f}, {r['qy']:.4f}, {r['qz']:.4f})"
          f" | grasp ({r['grasp x (mm)']:6.1f}, {r['grasp y (mm)']:6.1f}, {r['grasp z (mm)']:6.1f}) from {r['approach side']} side,"
          f" approach ({ax[0]:.3f}, {ax[1]:.3f}, {ax[2]:.3f}), tilt {r['approach tilt (deg)']:.0f} deg")
print()
show_pose_table(best[["rank", "conf score", "x (mm)", "y (mm)", "z (mm)", "rx (deg)", "ry (deg)", "rz (deg)",
                      "qw", "qx", "qy", "qz", "grasp x (mm)", "grasp y (mm)", "grasp z (mm)", "approach (cam)",
                      "approach side", "approach tilt (deg)", "hole blocked", "ring blocked", "occluded", "support",
                      "fit error (mm)"]], digits=3)

def gripper_circles(rows, frame):
    # draw the finger (yellow) and gripper body (light blue) circles at each hole entrance
    def draw(dr):
        K = frame["K"]
        for _, r in rows.iterrows():
            e = np.array([r["grasp x (mm)"], r["grasp y (mm)"], r["grasp z (mm)"]])
            ax = -np.asarray(r["approach (cam)"], float)
            a = np.cross(ax, [1.0, 0, 0]) if abs(ax[0]) < 0.9 else np.cross(ax, [0, 1.0, 0])
            a /= np.linalg.norm(a)
            b = np.cross(ax, a)
            ang = np.linspace(0, 2 * np.pi, 73)
            for rad, col in ((R_FINGER, (255, 230, 0)), (R_BODY, (80, 200, 255))):
                pts = e + rad * (np.cos(ang)[:, None] * a + np.sin(ang)[:, None] * b)
                p = pts @ K.T
                dr.line(list(zip(p[:, 0] / p[:, 2], p[:, 1] / p[:, 2])), fill=col, width=2)
    return draw

keep = set(best.index)
img = render_2d(fc["rgb"], fc["K"], [d if n in keep else None for n, d in enumerate(cdets)], model, None,
                labels="all", title=f"capture view {cv}: {len(best)} selected (on top + confidence + hole free)")
display(img.resize((1000, int(1000 * img.height / img.width))))
display(surface_overlay(fc, cdets, list(best.index), f"{len(best)} selected: CAD surface vs depth, finger (yellow) / gripper body (blue)",
                        draw=gripper_circles(best, fc)))
if dropped:
    display(surface_overlay(fc, cdets, dropped, "dropped from the previous top 10: hole or gripper ring blocked, tilt, ...",
                            draw=gripper_circles(full.loc[dropped], fc)))

### 전체 결과를 CSV로 저장

모든 장면의 출력 pose를 run 폴더의 `poses_6d.csv`에 저장합니다(로봇 쪽 코드나 다른 도구에서 읽기용). 한 줄이 부품 하나이고, depth 확인 값과 pick 순서도 들어 있습니다.

In [ ]:
def export_poses(run_dir, frames, idx, scene_has_gt):
    tabs = []
    for (s, view), ds in sorted(load_run(run_dir, cfg.refine).items()):
        if not ds:
            continue
        f = load_frame(frames, idx[(s, view)])
        tabs.append(pose_rows(ds, f["objects_all"] if scene_has_gt else None, f).reset_index().assign(scene=s, view=view))
    df = pd.concat(tabs, ignore_index=True)
    df = df[["scene", "view"] + [c for c in df.columns if c not in ("scene", "view")]]
    df["hub axis (cam)"] = df["hub axis (cam)"].map(lambda a: " ".join(f"{x:.4f}" for x in a))
    out = Path(run_dir) / "poses_6d.csv"
    df.to_csv(out, index=False, float_format="%.4f")
    n = df.view.nunique()
    print(f"{out}: {len(df)} parts in {n} frames, {int(df.confident.sum())} confident, "
          f"{int(df.pickable.sum())} pickable ({df.pickable.sum() / n:.1f} per frame)")
    return df

all_real = export_poses(POSE_RUN, real, ridx, True)
all_cap = export_poses(CAP_MAIN, cap, cidx, False)
for run_dir, frames, idx in ((POSE_RUN, real, ridx), (CAP_MAIN, cap, cidx)):
    tops = []
    for (s, view), ds in sorted(load_run(run_dir, cfg.refine).items()):
        if ds:
            f = load_frame(frames, idx[(s, view)])
            tops.append(top_picks(pose_rows(ds, frame=f), ds, f)[0].reset_index().assign(scene=s, view=view))
    tops = pd.concat(tops, ignore_index=True)
    tops = tops[["scene", "view", "rank"] + [c for c in tops.columns if c not in ("scene", "view", "rank")]]
    for col in ("hub axis (cam)", "approach (cam)"):
        tops[col] = tops[col].map(lambda a: " ".join(f"{x:.4f}" for x in a))
    tops.to_csv(Path(run_dir) / f"pick_top{TOP_K}.csv", index=False, float_format="%.4f")
    print(f"{Path(run_dir) / f'pick_top{TOP_K}.csv'}: up to {TOP_K} per frame (on top + confidence + hole free), "
          f"{len(tops)} rows, {len(tops) / tops.view.nunique():.1f} per frame")
for name, g in (("confident", all_real[all_real.confident]), ("pickable", all_real[all_real.pickable])):
    print(f"Photoneo 70, {name} vs pseudo-GT ({len(g)}): centre err median {g['GT centre err (mm)'].median():.2f} mm, "
          f"hub axis err median {g['GT hub axis err (deg)'].median():.2f} deg, ADD-S < 1 mm {(g['GT ADD-S (mm)'] < 1).mean():.1%}")

### 정확도 요약: 단계별로 걸러질 때 무엇이 남는가

위의 선택 단계(출력 → confident → 맨 위 → 집기 가능 → 장당 상위 10개)마다 정확도와 품질을 표로 정리합니다.

- **표 A (Photoneo 70장, 의사 정답과 비교):** ADD-S, 위치 오차, 허브 축 오차와 함께 **집는 점 오차**(예측 pose의 구멍 입구 중심과 의사 정답 pose의 구멍 입구 중심 사이 거리)를 봅니다.
  내경 그리퍼에는 이 값이 가장 직접적입니다. **side flipped**는 허브/판 중 어느 면이 위인지를 반대로 읽은 경우입니다(집는 점이 약 21mm 틀어짐).
  의사 정답 자체의 불확실성(위치 약 0.5mm, 기울기 1~2°)보다 작은 차이는 구분할 수 없습니다.
- **표 B (정답 없이 잰 품질, 두 데이터):** 맞춤 오차, depth 모순(violation), 덮임(occluded), 접근 기울기, 구멍과 링 막힘입니다. 가득 찬 bin은 정답이 없어서 이 표로만 판단합니다.
- **표 C:** 가득 찬 bin 장면별로 집을 수 있는 부품 수입니다.

In [ ]:
def entry_of(R, t):
    # hole entrance centre on the camera-facing side (same rule as grasp_check) and which side that is
    R, t = np.asarray(R, float), np.asarray(t, float)
    side = 1.0 if R[:, 2] @ t < 0 else -1.0
    z = model.xyz_min[2] + model.xyz_size[2] if side > 0 else model.xyz_min[2]
    return R @ np.array([0.0, 0.0, z]) + t, side

def eval_sets(run_dir, frames, idx, has_gt):
    out = []
    for (s, view), ds in sorted(load_run(run_dir, cfg.refine).items()):
        if not ds:
            continue
        f = load_frame(frames, idx[(s, view)])
        objs = f["objects_all"] if has_gt else None
        best, full = top_picks(pose_rows(ds, objs, f), ds, f)
        full = full.assign(selected=full.index.isin(best.index), view=view)
        if has_gt:
            gid = {int(i): k for k, i in enumerate(objs["inst_id"])}
            ge, flip = [], []
            for n in full.index:
                k = gid.get(ds[n]["gt_inst_id"]) if ds[n].get("gt_inst_id") is not None else None
                if k is None:
                    ge.append(np.nan); flip.append(np.nan)
                    continue
                (e1, s1), (e0, s0) = entry_of(ds[n]["R"], ds[n]["t"]), entry_of(objs["R"][k], objs["t"][k])
                ge.append(float(np.linalg.norm(e1 - e0))); flip.append(float(s1 != s0))
            full["GT grasp point err (mm)"], full["side flipped"] = ge, flip
        out.append(full)
    return pd.concat(out)

STAGES = [("all outputs", lambda d: d.index == d.index), ("confident", lambda d: d.confident.astype(bool)),
          ("on top (pickable)", lambda d: d.pickable.astype(bool)), ("graspable", lambda d: d["grasp ok"].fillna(False).astype(bool)),
          (f"selected (top {TOP_K} / frame)", lambda d: d.selected.astype(bool))]

def med_p95(x, fmt="{:.2f}"):
    x = pd.Series(x, dtype=float).dropna()
    return f"{fmt.format(x.median())} / {fmt.format(x.quantile(0.95))}" if len(x) else "-"

def table_gt(d):
    n_frames = d.view.nunique()
    rows = {}
    for name, sel in STAGES:
        m = d[sel(d)]
        g = m[m["GT ADD-S (mm)"].notna()]
        rows[name] = {"parts": len(m), "per frame": round(len(m) / n_frames, 1),
                      "matched to pseudo-GT": len(g), "no pseudo-GT part (false detection)": len(m) - len(g),
                      "ADD-S < 1 mm": f"{(g['GT ADD-S (mm)'] < 1).mean():.1%}" if len(g) else "-",
                      "wrong (ADD-S >= 1 mm)": int((g["GT ADD-S (mm)"] >= 1).sum()),
                      "centre err median / p95 (mm)": med_p95(g["GT centre err (mm)"]),
                      "hub axis err median / p95 (deg)": med_p95(g["GT hub axis err (deg)"]),
                      "grasp point err median / p95 (mm)": med_p95(g["GT grasp point err (mm)"]),
                      "grasp point err max (mm)": round(float(g["GT grasp point err (mm)"].max()), 2) if len(g) else "-",
                      "side flipped": int(g["side flipped"].sum())}
    return pd.DataFrame(rows)

def table_free(d):
    n_frames = d.view.nunique()
    rows = {}
    for name, sel in STAGES:
        m = d[sel(d)]
        c = m[m.confident.astype(bool)]
        rows[name] = {"parts": len(m), "per frame": round(len(m) / n_frames, 1),
                      "fit error median / p95 (mm)": med_p95(m["fit error (mm)"]),
                      "fitness median": round(float(m.fitness.median()), 3) if len(m) else "-",
                      "support median": round(float(m.support.median()), 2) if len(m) else "-",
                      "violation median / p95": med_p95(m.violation, "{:.3f}"),
                      "occluded median": round(float(m.occluded.median()), 2) if len(m) else "-",
                      "approach tilt median / p95 (deg)": med_p95(c["approach tilt (deg)"], "{:.0f}") if "approach tilt (deg)" in c else "-",
                      "hole blocked median / p95": med_p95(c["hole blocked"], "{:.3f}") if "hole blocked" in c else "-",
                      "ring blocked median / p95": med_p95(c["ring blocked"], "{:.3f}") if "ring blocked" in c else "-"}
    return pd.DataFrame(rows)

ev_real = eval_sets(POSE_RUN, real, ridx, True)
ev_cap = eval_sets(CAP_MAIN, cap, cidx, False)
print(f"A. Photoneo {len(real)} frames vs pseudo-GT ({POSE_RUN})")
display(table_gt(ev_real))
print(f"B. label-free quality: Photoneo {len(real)} frames ({POSE_RUN})")
display(table_free(ev_real))
print(f"B. label-free quality: full bin, {len(cap)} frames, no GT ({CAP_MAIN})")
display(table_free(ev_cap))
k = ev_cap.groupby("view")
print("C. full bin, per frame")
display(pd.DataFrame({"outputs": k.size(), "confident": k.confident.sum(), "on top": k.pickable.sum(),
                      "graspable": k["grasp ok"].sum(), f"selected (<= {TOP_K})": k.selected.sum()}).astype(int).T)